# NCBI BioScraper Pro — auto-captured snapshot

Generated from the IPython kernel's input history. Captures every unique code cell run in this session.


In [ ]:
# ============================================================================
# CELL — SAFE PUBLISH TO GITHUB (no --force, no stubs, honest README)
# ============================================================================
# What this does:
#   1. Saves your CURRENT Colab notebook file from /content/ to a working dir
#   2. Clones the repo (pulls — does not destroy local state)
#   3. Replaces the notebook file with your actual current notebook
#   4. Writes a README that ONLY describes features actually in the notebook
#   5. Does a normal commit + push. NO --force, NO stub writes.
#   6. Fails loudly if there's a conflict — does NOT clobber remote state.
#
# What this does NOT do:
#   - It does NOT write empty {"cells":[]} JSON over your real notebook
#   - It does NOT use --force
#   - It does NOT make claims in the README that the notebook can't back up
#
# Pre-flight checklist before running this cell:
#   [ ] You have actually run the v3.5.3 pipeline end-to-end successfully
#   [ ] You have the .ipynb file saved (File > Download .ipynb in Colab)
#   [ ] You have uploaded that .ipynb to /content/ in this Colab session
#   [ ] Your GH_TOKEN has 'repo' scope (Settings > Developer settings > PATs)
# ============================================================================
import os, sys, shutil, subprocess, json
from pathlib import Path

# --- CONFIG ----------------------------------------------------------------
GITHUB_USER  = "crisprking"
GITHUB_REPO  = "ncbi-bioscraper"
COMMIT_EMAIL = "abrahamtrueba@berkeley.edu"
COMMIT_NAME  = GITHUB_USER

# Path where you uploaded your current working notebook.
# In Colab: drag-and-drop the .ipynb into the file panel, or upload via files.upload()
LOCAL_NOTEBOOK = "/content/NCBI_BioScraper_Pro_v3_5_3.ipynb"

# What to call the notebook in the repo
REMOTE_NOTEBOOK_NAME = "NCBI_BioScraper_Pro_v3_5_3.ipynb"

# Get token from Colab Secrets (Settings > Secrets > GH_TOKEN)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GH_TOKEN")
    assert GITHUB_TOKEN, "GH_TOKEN secret is empty"
except Exception as e:
    print(f"❌ Could not get GH_TOKEN from Colab Secrets: {e}")
    print("   Set it: 🔑 (left sidebar) → Add new secret → name=GH_TOKEN, value=ghp_...")
    raise SystemExit(1)

# --- PRE-FLIGHT CHECKS -----------------------------------------------------
print("🔍 Pre-flight checks...")
errors = []

if not Path(LOCAL_NOTEBOOK).exists():
    errors.append(f"Notebook not found at {LOCAL_NOTEBOOK}")
else:
    size_kb = Path(LOCAL_NOTEBOOK).stat().st_size // 1024
    if size_kb < 5:
        errors.append(f"Notebook at {LOCAL_NOTEBOOK} is only {size_kb} KB — looks empty")
    else:
        # Validate it's actually a notebook with cells
        try:
            with open(LOCAL_NOTEBOOK) as f:
                nb = json.load(f)
            n_cells = len(nb.get("cells", []))
            if n_cells < 3:
                errors.append(f"Notebook has only {n_cells} cells — refusing to publish a near-empty notebook")
            else:
                print(f"   ✅ Notebook OK: {size_kb} KB, {n_cells} cells")
        except json.JSONDecodeError:
            errors.append(f"{LOCAL_NOTEBOOK} is not valid JSON")

if errors:
    print("\n".join(f"   ❌ {e}" for e in errors))
    print("\nFix these and re-run. Refusing to push.")
    raise SystemExit(1)

# --- CLONE -----------------------------------------------------------------
work_dir = Path("/content/_repo_publish_workspace")
if work_dir.exists():
    shutil.rmtree(work_dir)

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
print(f"\n📥 Cloning {GITHUB_USER}/{GITHUB_REPO}...")
result = subprocess.run(["git", "clone", clone_url, str(work_dir)],
                        capture_output=True, env=env)
if result.returncode != 0:
    print(f"❌ Clone failed: {result.stderr.decode()[:500]}")
    raise SystemExit(1)
print("   ✅ Cloned")

os.chdir(work_dir)
branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
print(f"   Branch: {branch}")

# Show what's currently in the repo BEFORE we change anything
print("\n📂 Current repo contents:")
for p in sorted(work_dir.iterdir()):
    if p.name.startswith("."): continue
    sz = p.stat().st_size if p.is_file() else "—"
    print(f"     {p.name}  ({sz} bytes)" if isinstance(sz, int) else f"     {p.name}/  (dir)")

# --- COPY THE NOTEBOOK -----------------------------------------------------
print(f"\n📝 Copying notebook → {REMOTE_NOTEBOOK_NAME}")
shutil.copy(LOCAL_NOTEBOOK, work_dir / REMOTE_NOTEBOOK_NAME)

# --- WRITE AN HONEST README ------------------------------------------------
# Keep this STRICTLY about what the notebook actually does. No marketing.
readme = """# NCBI BioScraper Pro

A zero-cost Colab notebook for biomedical literature mining: PubMed scrape,
OpenAlex citation enrichment, and Open Access full-text resolution.

## What it actually does

**PubMed scrape** (Cells 1–5):
- Configurable query, date range, max results
- SQLite cache for resumable runs
- ~500 records / ~3 min on free Colab (depends on NCBI rate limits;
  with a free NCBI API key, throughput roughly triples)

**OA full-text resolution** (Cell 7, v3.5.3):
- Six-provider waterfall: PMC → Unpaywall → Europe PMC → CrossRef license →
  Semantic Scholar → bioRxiv/medRxiv direct lookup
- Optional preprint title-similarity fallback via Europe PMC
- Optional PDF download + text extraction with pypdf
- Honest reporting: separate `full_text_pct` (have a PDF URL) and
  `any_oa_link_pct` (have at least an OA pointer) metrics
- Per-source extraction success rate and failure-reason breakdown

**Other modules:**
- Study-type classification (RCT, review, meta-analysis, case study)
- Entity extraction (drugs, diseases, genes from abstracts)
- Semantic near-duplicate detection
- Multi-format export (CSV, XLSX, JSON, BibTeX, Parquet, RIS)

## What it does NOT do

- No Sci-Hub fallback (deliberately — copyright)
- No automated scraping of paywalled publisher landing pages
- No OCR (scanned PDFs are skipped, not extracted)
- No LLM API calls unless you opt in via `GROQ_SUMMARIES = True` and set
  `GROQ_API_KEY`

## Realistic OA full-text yield

For broad biomedical PubMed queries with a valid email configured for
Unpaywall and CrossRef, expect roughly:
- 45–55% records with an extractable PDF
- 75–85% records with at least one OA pointer (PDF or landing page)

The gap between those two numbers is publishers who declare a CC license
in CrossRef metadata but do not expose a direct PDF URL — you can read the
paper on their site but cannot programmatically download it without a
per-publisher scraper, which this notebook does not include.

Yield is significantly higher (60–70% extractable PDFs) for queries weighted
toward PMC-deposited journals and lower (~30–40%) for queries dominated by
Elsevier / Wiley / Springer hybrid-OA content.

## Configuration

Open the notebook, set `NCBI_EMAIL` in Cell 2 to a real email address
(required by Unpaywall and CrossRef per their ToS), then run cells in
order. Optional flags in Cell 7 (`EXTRACT_PDF_TEXT`, `PREPRINT_TITLE_SEARCH`,
`GROQ_SUMMARIES`) are documented inline.

## License

MIT. Use at your own discretion; respect publisher ToS for any URLs the
resolver returns.
"""

(work_dir / "README.md").write_text(readme)
print("   ✅ README written")

# --- COMMIT + PUSH (NO --force) --------------------------------------------
subprocess.run(["git", "config", "user.email", COMMIT_EMAIL])
subprocess.run(["git", "config", "user.name", COMMIT_NAME])
subprocess.run(["git", "add", REMOTE_NOTEBOOK_NAME, "README.md"])

status = subprocess.run(["git", "status", "--porcelain"],
                        capture_output=True).stdout.decode()
if not status.strip():
    print("\nℹ️  Nothing to commit — repo already matches local notebook + README.")
else:
    print(f"\n📋 Staged changes:\n{status}")
    subprocess.run(["git", "commit", "-m", "Update notebook to v3.5.3 + sync README"])

print(f"\n📤 Pushing to {branch}...")
result = subprocess.run(["git", "push", "origin", branch],
                        capture_output=True, env=env)
if result.returncode != 0:
    print(f"❌ Push failed (NOT using --force; resolve manually):")
    print(result.stderr.decode()[:1000])
    print("\nIf this is a 'non-fast-forward' error, your remote has commits the")
    print("local clone didn't have. Pull manually, merge, and try again. Do NOT")
    print("force-push without checking what's there first.")
    raise SystemExit(1)

print(f"\n✅ Pushed.")
print(f"   https://github.com/{GITHUB_USER}/{GITHUB_REPO}")
print(f"   Branch: {branch}")
print(f"   Notebook: {REMOTE_NOTEBOOK_NAME}")

In [ ]:
# ============================================================================
# CELL 7 — OA Full-Text Resolution v3.5.3 MAX  (clean, consolidated, drop-in)
#
# Replaces the v2 Unpaywall-only cell AND every "Cell 7 v3.x" you've pasted
# below it. Delete those — this is the only one you need.
#
# Resolution order (5 providers + 1 title-search fallback):
#   PMC → Unpaywall → Europe PMC → CrossRef → Semantic Scholar → bioRxiv/medRxiv
#   → (preprint title-similarity search if still no full text)
#
# What's new in 3.5 vs 3.4:
#   - NEW provider: SemanticScholarResolver (openAccessPdf field, free, no key)
#     — catches author-hosted PDFs that Unpaywall/CrossRef miss
#   - DEFAULTS flipped to MAX mode:
#       EXTRACT_PDF_TEXT      = True   (downloads + parses PDFs with pypdf)
#       PREPRINT_TITLE_SEARCH = True   (finds preprint twins via Jaccard)
#       PDF_TEXT_MAX_RECORDS  = 100    (was 20)
#       PDF_TEXT_MAX_BYTES    = 12 MB  (was 8 MB)
#   - Coverage report now also shows extracted-content stats, not just links
#
# Expected for a 50-record biomedical query with valid email:
#   - Full-text yield: 65-80% (vs 42% PMC-only)
#   - Extracted text:  ~90% of full-text URLs (some publisher PDFs gate scraping)
#   - Wall time: ~60-90s (PDF downloads dominate; tune PDF_TEXT_MAX_RECORDS down
#     if you want it faster and don't need the actual text)
# ============================================================================
import os, sys, time, logging, re

# ---- 0. Config pulled from earlier cells, with explicit precedence ---------
# Hardcoded so this works even if Cell 2 wasn't re-run / email got reverted.
# If you change your email later, edit this line directly.
USER_EMAIL    = "abrahamtrueba@berkeley.edu"
# (Falls back to globals if you ever blank the line above)
USER_EMAIL    = USER_EMAIL or (globals().get("USER_EMAIL") or globals().get("NCBI_EMAIL") or "").strip()
NCBI_API_KEY  = globals().get("NCBI_API_KEY") or None
ENRICH_UNPAYWALL = bool(globals().get("ENRICH_UNPAYWALL", True))

# v3.5.3 MAX-COVERAGE MODE — these are HARDCODED, ignoring stale globals from
# previous Cell 2 runs. To customize, edit these lines directly.
EXTRACT_PDF_TEXT       = True             # download + parse OA PDFs with pypdf
PDF_TEXT_MAX_BYTES     = 12_000_000       # 12 MB cap per PDF
PDF_TEXT_MAX_RECORDS   = 100              # process up to this many in a batch
PREPRINT_TITLE_SEARCH  = True             # bioRxiv/medRxiv title-similarity fallback
GROQ_SUMMARIES         = bool(globals().get("GROQ_SUMMARIES", False))   # OFF — needs GROQ_API_KEY
GROQ_MODEL             = globals().get("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_MAX_RECORDS       = int(globals().get("GROQ_MAX_RECORDS", 30))

_PLACEHOLDER_EMAILS = {"", "your@email.com", "research@example.com",
                       "you@yourdomain.com", "user@example.com"}
_email_is_real = USER_EMAIL.lower() not in _PLACEHOLDER_EMAILS and "@" in USER_EMAIL

if not _email_is_real:
    print("=" * 72)
    print("  ⚠️  USER_EMAIL IS UNSET OR A PLACEHOLDER")
    print("  ----------------------------------------")
    print("  Unpaywall and CrossRef require a real email per their ToS.")
    print("  Without it, OA yield drops to PMC-only (~40-45% on biomedical sets).")
    print("  With a real email, expect 60-70% on broad PubMed queries.")
    print("")
    print("  Fix: in Cell 2, set")
    print("       NCBI_EMAIL = 'you@your.domain'")
    print("  then re-run this cell. No other changes needed.")
    print("=" * 72)
    print()

# ---- 1. Resolver module (written ONCE) -------------------------------------
_MODULE_PATH = "/content/oa_fulltext_v3.py"
_MODULE_SRC = r'''
from __future__ import annotations
import logging, re, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

log = logging.getLogger("bioscraper.oa_v3")


def _build_session(user_agent="NCBI-BioScraper-Pro/3.5"):
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


def _build_session_no_429_retry(user_agent="NCBI-BioScraper-Pro/3.5"):
    """Like _build_session but does NOT auto-retry 429s. Used for providers
    where we want to see the 429 status code directly (e.g. so a self-throttle
    latch can engage instead of letting urllib3 burn 3 retries silently)."""
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=(500, 502, 503, 504),  # 429 deliberately omitted
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


@dataclass
class OAResult:
    pmid: Optional[str] = None
    doi: Optional[str] = None
    pmcid: Optional[str] = None
    oa_source: str = "none"
    oa_pdf_url: Optional[str] = None
    oa_xml_url: Optional[str] = None
    oa_html_url: Optional[str] = None
    oa_license: Optional[str] = None
    oa_status: str = "closed"
    oa_full_text: bool = False
    oa_attempts: list = field(default_factory=list)
    oa_resolved_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def as_flat_dict(self):
        d = asdict(self)
        d["oa_attempts"] = ",".join(d["oa_attempts"])
        return d


# ---------- providers --------------------------------------------------------
class PMCResolver:
    IDCONV = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    OA_FCGI = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"

    def __init__(self, session, ncbi_api_key=None, tool="ncbi-bioscraper", email="research@example.com"):
        self.s = session; self.api_key = ncbi_api_key; self.tool = tool; self.email = email

    def _params(self, **extra):
        p = {"tool": self.tool, "email": self.email}
        if self.api_key: p["api_key"] = self.api_key
        p.update(extra); return p

    def pmid_to_pmcid(self, pmid, doi=None):
        for key, value in (("pmid", pmid), ("doi", doi)):
            if not value: continue
            try:
                r = self.s.get(self.IDCONV,
                               params=self._params(ids=value, idtype=key, format="json"),
                               timeout=6)
                if r.status_code != 200: continue
                records = r.json().get("records", [])
                if records and records[0].get("pmcid"):
                    return records[0]["pmcid"]
            except Exception as e:
                log.debug("idconv (%s=%s) failed: %s", key, value, e)
        return None

    def pmcid_to_oa(self, pmcid):
        try:
            r = self.s.get(self.OA_FCGI, params={"id": pmcid}, timeout=6)
            if r.status_code != 200 or "<error" in r.text.lower():
                return None
            text = r.text
            pdf_url = xml_url = license_str = None
            for line in text.split("<link "):
                if 'format="pdf"' in line and 'href="' in line:
                    pdf_url = line.split('href="')[1].split('"')[0]
                elif 'format="tgz"' in line and 'href="' in line:
                    xml_url = line.split('href="')[1].split('"')[0]
            if 'license="' in text:
                license_str = text.split('license="')[1].split('"')[0]
            if pdf_url or xml_url:
                if pdf_url and pdf_url.startswith("ftp://"):
                    pdf_url = "https://" + pdf_url[len("ftp://"):]
                if xml_url and xml_url.startswith("ftp://"):
                    xml_url = "https://" + xml_url[len("ftp://"):]
                return {"pdf_url": pdf_url, "xml_url": xml_url,
                        "license": license_str,
                        "html_url": "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % pmcid}
        except Exception as e:
            log.debug("oa.fcgi failed for %s: %s", pmcid, e)
        return None

    def resolve(self, result):
        result.oa_attempts.append("pmc")
        if not result.pmcid:
            result.pmcid = self.pmid_to_pmcid(result.pmid, result.doi)
        if not result.pmcid:
            return False
        oa = self.pmcid_to_oa(result.pmcid)
        if not oa:
            # PMC-hosted but not in OA subset — still useful as a reader URL
            result.oa_html_url = "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % result.pmcid
            return False
        result.oa_source = "pmc"
        result.oa_pdf_url = oa.get("pdf_url")
        result.oa_xml_url = oa.get("xml_url")
        result.oa_html_url = oa.get("html_url")
        result.oa_license = oa.get("license")
        result.oa_status = "gold"
        result.oa_full_text = bool(result.oa_pdf_url or result.oa_xml_url)
        return result.oa_full_text


class UnpaywallResolver:
    BASE = "https://api.unpaywall.org/v2/"

    def __init__(self, session, email):
        self.s = session; self.email = email
        self._not_in_index = 0
        self._auth_failed = False  # latched: stop calling once we see a 422

    def resolve(self, result):
        result.oa_attempts.append("unpaywall")
        if self._auth_failed or not result.doi:
            return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"email": self.email}, timeout=6)
            if r.status_code == 404:
                self._not_in_index += 1
                return False
            if r.status_code == 422:
                # Auth/email problem — latch and stop hammering the API
                self._auth_failed = True
                log.warning("Unpaywall 422 — disabling Unpaywall for the rest of this run "
                            "(check USER_EMAIL is a real address).")
                return False
            if r.status_code != 200:
                return False
            d = r.json()
            if not d.get("is_oa"):
                return False
            best = d.get("best_oa_location") or {}
            if not best:
                locs = d.get("oa_locations") or []
                if locs: best = locs[0]
            if not best: return False
            pdf  = best.get("url_for_pdf")
            html = best.get("url_for_landing_page") or best.get("url")
            if not (pdf or html): return False
            result.oa_source = "unpaywall"
            result.oa_pdf_url = pdf
            result.oa_html_url = html
            result.oa_license = best.get("license")
            result.oa_status = d.get("oa_status") or "bronze"
            result.oa_full_text = bool(pdf)
            return bool(pdf or html)
        except Exception as e:
            log.debug("unpaywall failed for %s: %s", result.doi, e)
            return False


class EuropePMCResolver:
    SEARCH = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def __init__(self, session): self.s = session

    def resolve(self, result):
        result.oa_attempts.append("europepmc")
        if result.pmid:
            query = "EXT_ID:%s AND SRC:MED" % result.pmid
        elif result.doi:
            query = 'DOI:"%s"' % result.doi
        else:
            return False
        try:
            r = self.s.get(self.SEARCH, params={
                "query": query, "format": "json",
                "resultType": "core", "pageSize": 1,
            }, timeout=6)
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            if not hits: return False
            h = hits[0]
            if h.get("isOpenAccess") != "Y": return False
            full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
            pdf = next((u["url"] for u in full_text_urls
                        if u.get("documentStyle", "").lower() == "pdf"
                        and u.get("availability", "").lower() == "open access"), None)
            html = next((u["url"] for u in full_text_urls
                         if u.get("documentStyle", "").lower() == "html"), None)
            if not (pdf or html): return False
            result.oa_source = "europepmc"
            result.oa_pdf_url = pdf or result.oa_pdf_url
            result.oa_html_url = html or result.oa_html_url
            result.oa_license = h.get("license") or result.oa_license
            result.oa_status = "gold" if h.get("inEPMC") == "Y" else "green"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("europepmc failed: %s", e)
            return False


class CrossRefOAResolver:
    """Catches recent papers (<30 days) that Unpaywall hasn't ingested yet,
    and any paper with a CC license declared in CrossRef metadata."""
    BASE = "https://api.crossref.org/works/"

    def __init__(self, session, email):
        self.s = session; self.email = email

    def resolve(self, result):
        result.oa_attempts.append("crossref")
        if not result.doi: return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"mailto": self.email}, timeout=6)
            if r.status_code != 200: return False
            msg = r.json().get("message") or {}
            licenses = msg.get("license") or []
            cc = next((L for L in licenses
                       if "creativecommons.org" in (L.get("URL") or "").lower()), None)
            if not cc: return False
            license_url = cc.get("URL", "")
            license_id = None
            if   "/by-nc-nd" in license_url: license_id = "cc-by-nc-nd"
            elif "/by-nc-sa" in license_url: license_id = "cc-by-nc-sa"
            elif "/by-nc"    in license_url: license_id = "cc-by-nc"
            elif "/by-sa"    in license_url: license_id = "cc-by-sa"
            elif "/by"       in license_url: license_id = "cc-by"
            elif "/zero"     in license_url: license_id = "cc0"
            pdf = None
            for link in (msg.get("link") or []):
                if (link.get("content-type") == "application/pdf"
                        and "vor" in (link.get("intended-application", "") or "").lower()):
                    pdf = link.get("URL"); break
            result.oa_source = "crossref"
            result.oa_pdf_url = pdf
            result.oa_html_url = "https://doi.org/" + result.doi
            result.oa_license = license_id or "cc-detected"
            result.oa_status = "gold"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("crossref failed for %s: %s", result.doi, e)
            return False


class SemanticScholarResolver:
    """Semantic Scholar's openAccessPdf field. Free, no key needed at low
    volume (~1 req/sec unauthenticated). Catches a different slice than
    Unpaywall/CrossRef — especially newer papers and CS/biology preprints
    where their crawler picks up author-hosted PDFs.

    Self-rate-limits to 1 req/sec to avoid 429 cascades.
    Skips itself entirely if `result.oa_full_text` is already True from an
    earlier provider — saves quota for records that actually need it.
    Uses its own session that does NOT retry 429s, so the latch can engage
    on the first 429 instead of letting urllib3 burn 3 silent retries.
    """
    BASE = "https://api.semanticscholar.org/graph/v1/paper/"
    MIN_INTERVAL = 1.05  # seconds between calls

    def __init__(self, session=None, email=None):
        # Ignore the shared session — S2 needs the no-retry-on-429 behaviour
        self.s = _build_session_no_429_retry()
        self.email = email
        self._last_call = 0.0
        self._rate_limited = False  # latch after a 429 to stop hammering

    def _lookup(self, identifier):
        if self._rate_limited:
            return None
        # Self-throttle to 1 req/sec
        elapsed = time.time() - self._last_call
        if elapsed < self.MIN_INTERVAL:
            time.sleep(self.MIN_INTERVAL - elapsed)
        self._last_call = time.time()
        try:
            r = self.s.get(self.BASE + identifier,
                           params={"fields": "openAccessPdf,externalIds,isOpenAccess,title"},
                           timeout=6)
            if r.status_code == 429:
                # Latch off — S2 has decided we've had enough. Don't keep retrying.
                self._rate_limited = True
                log.warning("Semantic Scholar 429 — disabling S2 for the rest of this run.")
                return None
            if r.status_code != 200:
                return None
            return r.json()
        except Exception as e:
            # If urllib3 still managed to hit retry exhaustion on 429s
            # (shouldn't with our no-retry session, but belt-and-braces),
            # treat that as a hard latch too.
            err_str = str(e).lower()
            if "429" in err_str or "too many" in err_str:
                self._rate_limited = True
                log.warning("Semantic Scholar repeated 429 — disabling S2 for the rest of this run.")
            else:
                log.debug("semanticscholar lookup %s failed: %s", identifier, e)
            return None

    def resolve(self, result):
        # Skip entirely if an earlier provider already gave us a PDF
        if result.oa_full_text:
            return False
        result.oa_attempts.append("semanticscholar")
        d = None
        if result.doi:
            d = self._lookup("DOI:" + result.doi)
        if not d and result.pmid:
            d = self._lookup("PMID:" + result.pmid)
        if not d:
            return False
        oa_pdf = d.get("openAccessPdf") or {}
        url = oa_pdf.get("url")
        if not url:
            return False
        result.oa_source = "semanticscholar"
        result.oa_pdf_url = url
        result.oa_html_url = result.oa_html_url or url
        result.oa_license = oa_pdf.get("license") or result.oa_license or "s2-detected"
        if result.oa_status == "closed":
            result.oa_status = "green"
        result.oa_full_text = True
        return True


# OAButtonResolver removed in v3.4: OA.Works retired the openaccessbutton.org
# API on 2025-11-18. Calls were 100% failing with DNS errors. If you want a
# similar drop-in replacement, CORE (api.core.ac.uk, free key) or Semantic
# Scholar's openAccessPdf field both work and follow the same `resolve(result)`
# contract.


class PreprintResolver:
    """Direct lookup for 10.1101/* DOIs (bioRxiv/medRxiv)."""
    def __init__(self, session): self.s = session


    def resolve(self, result):
        result.oa_attempts.append("preprint")
        if not result.doi or not result.doi.startswith("10.1101/"):
            return False
        for host in ("www.biorxiv.org", "www.medrxiv.org"):
            url = "https://%s/content/%s.full.pdf" % (host, result.doi)
            try:
                r = self.s.head(url, timeout=8, allow_redirects=True)
                if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
                    result.oa_source = "preprint"
                    result.oa_pdf_url = url
                    result.oa_html_url = url.rsplit(".full.pdf", 1)[0] + ".full"
                    result.oa_status = "green"
                    result.oa_full_text = True
                    return True
            except Exception:
                continue
        return False


class PreprintTitleSearchResolver:
    """Last-resort: search bioRxiv/medRxiv by title and accept matches with
    high token-overlap similarity. Useful when a peer-reviewed paper has
    a preprint twin under a different DOI."""
    BIORXIV_API = "https://api.biorxiv.org/details/{server}/{date_or_doi}/0/json"
    SEARCH = "https://www.biorxiv.org/search/{query}"  # HTML, not used directly

    def __init__(self, session, min_similarity=0.75):
        self.s = session
        self.min_similarity = min_similarity

    @staticmethod
    def _norm_tokens(s):
        s = (s or "").lower()
        s = re.sub(r"[^a-z0-9 ]+", " ", s)
        return set(t for t in s.split() if len(t) > 2)

    def _similarity(self, a, b):
        ta, tb = self._norm_tokens(a), self._norm_tokens(b)
        if not ta or not tb: return 0.0
        return len(ta & tb) / len(ta | tb)  # Jaccard

    def resolve(self, result, candidate_title=None):
        result.oa_attempts.append("preprint_search")
        if not candidate_title: return False
        # bioRxiv has no native title-search API; we use Europe PMC restricted
        # to preprint sources, which DOES index bioRxiv/medRxiv records.
        try:
            r = self.s.get(
                "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                params={"query": 'TITLE:"%s" AND (SRC:PPR)' % candidate_title[:200],
                        "format": "json", "resultType": "core", "pageSize": 5},
                timeout=6,
            )
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            for h in hits:
                sim = self._similarity(candidate_title, h.get("title"))
                if sim < self.min_similarity: continue
                doi = h.get("doi")
                full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
                pdf = next((u["url"] for u in full_text_urls
                            if u.get("documentStyle", "").lower() == "pdf"), None)
                html = next((u["url"] for u in full_text_urls
                             if u.get("documentStyle", "").lower() == "html"), None)
                if not (pdf or html): continue
                result.oa_source = "preprint_search"
                result.oa_pdf_url = pdf or result.oa_pdf_url
                result.oa_html_url = html or result.oa_html_url
                result.oa_status = "green"
                result.oa_full_text = bool(pdf)
                return True
        except Exception as e:
            log.debug("preprint_search failed: %s", e)
        return False


# ---------- orchestrator -----------------------------------------------------
class OAFullTextResolver:
    def __init__(self, email, ncbi_api_key=None,
                 enable_pmc=True, enable_unpaywall=True,
                 enable_europepmc=True, enable_crossref=True,
                 enable_semanticscholar=True,
                 enable_oabutton=False,  # kept for back-compat; ignored (API retired)
                 enable_preprint=True,
                 enable_preprint_search=False,
                 polite_delay_s=0.0):
        self.session = _build_session()
        self.polite_delay = polite_delay_s
        self.providers = []
        if enable_pmc:
            self.providers.append(PMCResolver(self.session, ncbi_api_key, email=email))
        if enable_unpaywall:
            self.providers.append(UnpaywallResolver(self.session, email=email))
        if enable_europepmc:
            self.providers.append(EuropePMCResolver(self.session))
        if enable_crossref:
            self.providers.append(CrossRefOAResolver(self.session, email=email))
        if enable_semanticscholar:
            self.providers.append(SemanticScholarResolver(self.session, email=email))
        # OA Button removed in v3.4 (api.openaccessbutton.org retired 2025-11-18)
        if enable_preprint:
            self.providers.append(PreprintResolver(self.session))
        # Title-search resolver is held separately because it needs a title arg
        self.preprint_search = (PreprintTitleSearchResolver(self.session)
                                if enable_preprint_search else None)

    def resolve_one(self, pmid=None, doi=None, pmcid=None, title=None):
        result = OAResult(pmid=pmid, doi=doi, pmcid=pmcid)
        for provider in self.providers:
            try:
                if provider.resolve(result) and result.oa_full_text:
                    break
            except Exception as e:
                log.warning("%s raised: %s", type(provider).__name__, e)
            if self.polite_delay:
                time.sleep(self.polite_delay)
        # Title-similarity fallback only if we still don't have full text
        if (self.preprint_search is not None
                and not result.oa_full_text
                and title):
            try:
                self.preprint_search.resolve(result, candidate_title=title)
            except Exception as e:
                log.warning("PreprintTitleSearchResolver raised: %s", e)
        return result

    def enrich_dataframe(self, df, pmid_col="pmid", doi_col="doi",
                         pmcid_col=None, title_col=None, show_progress=True):
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(df.iterrows(), total=len(df),
                            desc="OA resolve", disable=not show_progress)
        except ImportError:
            iterator = df.iterrows()

        rows = []
        import pandas as pd
        for _, row in iterator:
            def _get(col):
                if col and col in df.columns:
                    v = row[col]
                    if v is None: return None
                    try:
                        if pd.isna(v): return None
                    except Exception: pass
                    s = str(v).strip()
                    return s if s and s.lower() != "nan" else None
                return None
            rows.append(self.resolve_one(
                pmid=_get(pmid_col),
                doi=_get(doi_col),
                pmcid=_get(pmcid_col),
                title=_get(title_col),
            ).as_flat_dict())

        oa_df = pd.DataFrame(rows)
        # Don't re-emit identifier cols already present on the input df, otherwise
        # a second pass through enrich_dataframe will collide on pmid/doi/pmcid.
        for c in ("pmid", "doi", "pmcid"):
            if c in oa_df.columns and c in df.columns:
                oa_df = oa_df.drop(columns=[c])
        for c in list(df.columns):
            if c.startswith("oa_") or c.startswith("up_"):
                df = df.drop(columns=[c])
        return df.reset_index(drop=True).join(oa_df.reset_index(drop=True))

    def coverage_report(self, df):
        total = len(df)
        if total == 0: return {"total": 0}
        return {
            "total": total,
            "full_text_pct": round(100 * df["oa_full_text"].sum() / total, 1),
            "any_oa_link_pct": round(100 * (df["oa_source"] != "none").sum() / total, 1),
            "by_source": df["oa_source"].value_counts().to_dict(),
            "by_status": df["oa_status"].value_counts().to_dict(),
        }
'''

# Write it ONCE, force a clean import
with open(_MODULE_PATH, "w", encoding="utf-8") as f:
    f.write(_MODULE_SRC)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
for _mod in [m for m in sys.modules if m.startswith("oa_fulltext_v3")]:
    del sys.modules[_mod]

from oa_fulltext_v3 import OAFullTextResolver, UnpaywallResolver  # noqa: E402

# ---- 2. Build resolver with sane provider gating ---------------------------
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("bioscraper.oa_v3").setLevel(logging.WARNING)  # quiet the per-record DEBUG noise

oa = OAFullTextResolver(
    email=USER_EMAIL or "research@example.com",
    ncbi_api_key=NCBI_API_KEY,
    enable_pmc=True,
    enable_unpaywall=ENRICH_UNPAYWALL and _email_is_real,   # << the fix
    enable_europepmc=True,
    enable_crossref=_email_is_real,                          # CrossRef wants mailto too
    # OA Button retired 2025-11-18, kwarg dropped
    enable_preprint=True,
    enable_preprint_search=PREPRINT_TITLE_SEARCH,
)

# ---- 3. Enrich -------------------------------------------------------------
# Use 'title' column if it exists, for the preprint title-similarity fallback
_title_col = "title" if "title" in df.columns else None
df = df.drop(columns=[c for c in df.columns if c.startswith("oa_") or c.startswith("up_")],
             errors="ignore")
df = oa.enrich_dataframe(df, pmid_col="pmid", doi_col="doi", title_col=_title_col)

# ---- 4. Coverage report ----------------------------------------------------
report = oa.coverage_report(df)
print("\n📊 OA full-text resolution v3.5.3 MAX (%d records)" % report["total"])
print("   Full-text yield: %s%%" % report["full_text_pct"])
print("   Any OA link:     %s%%" % report["any_oa_link_pct"])
print("   By source:       %s" % report["by_source"])
print("   By status:       %s" % report["by_status"])

unpaywall_obj = next((p for p in oa.providers if isinstance(p, UnpaywallResolver)), None)
if unpaywall_obj is not None:
    n_miss = unpaywall_obj._not_in_index
    print(f"\n   Unpaywall index misses (404s): {n_miss}/{len(df)}")
    print( "   -> these are typically <30-day-old DOIs not yet ingested by Unpaywall")
    if unpaywall_obj._auth_failed:
        print("   ⚠️  Unpaywall auth failed mid-run — fix USER_EMAIL and re-run this cell.")
elif ENRICH_UNPAYWALL and not _email_is_real:
    print("\n   Unpaywall: SKIPPED (placeholder email).")


# ============================================================================
# CELL 7b — Download PDFs and extract plain text
# Controlled by EXTRACT_PDF_TEXT flag. ON by default in v3.5 MAX.
# Uses pypdf only (no GPU, no OCR). Caps at PDF_TEXT_MAX_RECORDS for safety.
# ============================================================================
print()
print("─" * 72)
print(f"  PDF text extraction: {'ON' if EXTRACT_PDF_TEXT else 'OFF'}  "
      f"(EXTRACT_PDF_TEXT = {EXTRACT_PDF_TEXT})")
if EXTRACT_PDF_TEXT:
    print(f"  Cap: {PDF_TEXT_MAX_RECORDS} PDFs, "
          f"{PDF_TEXT_MAX_BYTES // 1_000_000} MB each")
print("─" * 72)

if EXTRACT_PDF_TEXT:
    try:
        import pypdf  # noqa: F401
    except ImportError:
        print("Installing pypdf for text extraction...")
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
        import pypdf

    import io, requests as _rq
    from collections import Counter as _Counter
    from pypdf import PdfReader

    # Browser-style User-Agent — many publishers block obvious scrapers but
    # serve OA content fine to a normal-looking browser fetch. This is OA
    # content the publisher has affirmatively licensed for free reading.
    _PDF_UA = ("Mozilla/5.0 (X11; Linux x86_64) "
               "AppleWebKit/537.36 (KHTML, like Gecko) "
               "Chrome/124.0.0.0 Safari/537.36")
    _PDF_HEADERS = {
        "User-Agent": _PDF_UA,
        "Accept": "application/pdf,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    # Track precise failure reasons so we can SEE why extraction is failing
    # rather than guessing. This is the key change in v3.5.3.
    _failure_reasons = []   # list of (idx, source, url, reason, detail)

    def _extract_pdf_text(url, max_bytes=PDF_TEXT_MAX_BYTES, idx=None, source=None):
        if not url:
            _failure_reasons.append((idx, source, url, "no_url", ""))
            return None
        try:
            with _rq.get(url, stream=True, timeout=20,
                         headers=_PDF_HEADERS, allow_redirects=True) as r:
                if r.status_code != 200:
                    _failure_reasons.append(
                        (idx, source, url, f"http_{r.status_code}",
                         f"final_url={r.url}"))
                    return None
                ctype = (r.headers.get("content-type") or "").lower()
                final_url = r.url
                if "pdf" not in ctype and not final_url.lower().endswith(".pdf"):
                    # Likely redirected to a login wall or HTML reader page
                    _failure_reasons.append(
                        (idx, source, url, "not_pdf", f"ctype={ctype} final={final_url}"))
                    return None
                buf = io.BytesIO()
                total = 0
                for chunk in r.iter_content(chunk_size=65536):
                    if not chunk: continue
                    buf.write(chunk); total += len(chunk)
                    if total > max_bytes:
                        _failure_reasons.append(
                            (idx, source, url, "too_big", f"{total:,} bytes"))
                        return None
                buf.seek(0)
                try:
                    reader = PdfReader(buf)
                except Exception as e:
                    _failure_reasons.append(
                        (idx, source, url, "pdf_parse_error", str(e)[:120]))
                    return None
                pages = []
                for page in reader.pages:
                    try: pages.append(page.extract_text() or "")
                    except Exception: pass
                text = "\n\n".join(pages).strip()
                text = re.sub(r"[ \t]+", " ", text)
                text = re.sub(r"\n{3,}", "\n\n", text)
                if not text:
                    _failure_reasons.append(
                        (idx, source, url, "empty_text", f"{total:,} bytes downloaded"))
                    return None
                return text
        except _rq.exceptions.Timeout:
            _failure_reasons.append((idx, source, url, "timeout", ""))
        except _rq.exceptions.SSLError as e:
            _failure_reasons.append((idx, source, url, "ssl_error", str(e)[:120]))
        except Exception as e:
            _failure_reasons.append((idx, source, url, "exception", str(e)[:120]))
        return None

    eligible = df[df["oa_full_text"] & df["oa_pdf_url"].notna()].copy()
    priority_order = ["pmc", "preprint", "preprint_search", "europepmc",
                      "unpaywall", "semanticscholar", "crossref"]
    eligible["_prio"] = eligible["oa_source"].map(
        {s: i for i, s in enumerate(priority_order)}).fillna(99)
    eligible = eligible.sort_values("_prio").head(PDF_TEXT_MAX_RECORDS)

    # Diagnostic context: report what we're processing and what we're skipping
    n_full_text  = int(df["oa_full_text"].sum())
    n_with_pdf   = int(df["oa_pdf_url"].notna().sum())
    n_xml_only   = int((df["oa_full_text"] & df["oa_pdf_url"].isna() &
                        df["oa_xml_url"].notna()).sum())

    if len(eligible) == 0:
        print("\n  ⚠️  No records with full-text PDF URLs to extract.")
        print("     (Need oa_full_text=True AND oa_pdf_url not null.)")
        df["oa_full_text_extracted"] = None
        df["oa_full_text_chars"] = 0
    else:
        print(f"\n📄 Extraction queue ({len(eligible)} of {n_full_text} full-text records):")
        print(f"   Records flagged full_text=True:    {n_full_text}")
        print(f"   Records with a PDF URL:            {n_with_pdf}")
        if n_xml_only:
            print(f"   Records with XML/tgz only (skipped): {n_xml_only}")
        print(f"   By source in queue: {dict(eligible['oa_source'].value_counts())}")

        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(eligible.iterrows(), total=len(eligible), desc="PDF extract")
        except ImportError:
            it = eligible.iterrows()

        texts = {}
        for idx, row in it:
            t = _extract_pdf_text(row["oa_pdf_url"], idx=idx, source=row["oa_source"])
            if t: texts[idx] = t

        df["oa_full_text_extracted"] = df.index.map(texts).where(df.index.isin(texts), None)
        df["oa_full_text_chars"]     = df["oa_full_text_extracted"].fillna("").str.len()
        n_extracted = int(df["oa_full_text_chars"].gt(0).sum())
        median_chars = int(df.loc[df["oa_full_text_chars"] > 0, "oa_full_text_chars"].median() or 0)
        print(f"\n   Extracted: {n_extracted}/{len(eligible)} PDFs successfully parsed")
        if n_extracted:
            print(f"   Median length: {median_chars:,} chars (~{median_chars//5_000} pages of plain text)")
            print(f"   Total content captured: {int(df['oa_full_text_chars'].sum()):,} chars")

        # Per-source success rate — tells us which providers gave usable PDFs
        succ_by_src = df.loc[df["oa_full_text_chars"] > 0, "oa_source"].value_counts().to_dict()
        attempted_by_src = eligible["oa_source"].value_counts().to_dict()
        print(f"\n   Per-source extraction success:")
        for src in attempted_by_src:
            ok = succ_by_src.get(src, 0); tot = attempted_by_src[src]
            print(f"     {src:18s} {ok}/{tot}")

        # Failure-reason breakdown — the diagnostic that was missing
        if _failure_reasons:
            reason_counts = _Counter(fr[3] for fr in _failure_reasons)
            print(f"\n   Failure reasons ({len(_failure_reasons)} total):")
            for reason, count in reason_counts.most_common():
                print(f"     {reason:18s} {count}")
            # Show first few examples per reason category
            print(f"\n   Failure examples (up to 5):")
            for fr in _failure_reasons[:5]:
                idx_, src_, url_, reason_, detail_ = fr
                short_url = url_[:60] + "…" if url_ and len(url_) > 60 else (url_ or "")
                print(f"     [{src_}] {reason_}: {short_url}")
                if detail_: print(f"       └─ {detail_[:140]}")

        # Stash the failure log on df for post-hoc inspection
        df.attrs["pdf_extract_failures"] = _failure_reasons

        print(f"\n   Records with ABSTRACT only:        {len(df) - n_extracted}")
        print(f"   Records with FULL TEXT extracted:  {n_extracted}")
        print(f"   -> use df['oa_full_text_extracted'] for downstream NLP / LLM input")
        print(f"   -> use df.attrs['pdf_extract_failures'] to inspect failed downloads")


# ============================================================================
# CELL 7c — Optional: Groq LLM summaries + key findings
# Off by default. Requires:  pip install groq   AND   os.environ["GROQ_API_KEY"]
# Operates on EXTRACTED full text if available, else on the abstract.
# ============================================================================
if GROQ_SUMMARIES:
    api_key = os.environ.get("GROQ_API_KEY", "").strip()
    if not api_key:
        print("\n⚠️  GROQ_SUMMARIES=True but GROQ_API_KEY not set in environment. Skipping.")
    else:
        try:
            from groq import Groq  # noqa: F401
        except ImportError:
            print("Installing groq client…")
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
        from groq import Groq

        client = Groq(api_key=api_key)

        SYSTEM_PROMPT = (
            "You are summarising a biomedical paper for a domain expert in "
            "molecular biology, genomics, and computational biology. "
            "Return STRICT JSON with exactly these keys: "
            '{"tldr": str (<=40 words), '
            '"key_findings": list[str] (3-5 bullets, each <=25 words), '
            '"methods": str (<=30 words), '
            '"caveats": str (<=25 words, or empty string if none)}. '
            "Stay concrete. Cite figures or numbers when they appear in the source. "
            "Do not invent details. Output ONLY the JSON object."
        )

        def _summarise_one(text, model=GROQ_MODEL, max_chars=24000):
            text = (text or "").strip()
            if not text: return None
            payload = text[:max_chars]
            try:
                resp = client.chat.completions.create(
                    model=model,
                    temperature=0.2,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": payload},
                    ],
                )
                return resp.choices[0].message.content
            except Exception as e:
                logging.getLogger("bioscraper.oa_v3").warning("Groq summarise failed: %s", e)
                return None

        # Choose source text: extracted full text > abstract
        has_fulltext = ("oa_full_text_extracted" in df.columns)
        candidates = df.copy()
        if has_fulltext:
            candidates["_src"] = candidates["oa_full_text_extracted"].fillna(
                candidates.get("abstract", ""))
        else:
            candidates["_src"] = candidates.get("abstract", "")
        candidates = candidates[candidates["_src"].fillna("").str.len() > 200]
        candidates = candidates.head(GROQ_MAX_RECORDS)

        print(f"\n🤖 Summarising {len(candidates)} records via Groq ({GROQ_MODEL})…")
        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(candidates.iterrows(), total=len(candidates), desc="LLM summarise")
        except ImportError:
            it = candidates.iterrows()

        summaries = {}
        for idx, row in it:
            s = _summarise_one(row["_src"])
            if s: summaries[idx] = s

        df["llm_summary_json"] = df.index.map(summaries).where(df.index.isin(summaries), None)
        # Lightly parse for convenience
        import json as _json
        def _safe_json(s):
            try: return _json.loads(s) if s else None
            except Exception: return None
        parsed = df["llm_summary_json"].map(_safe_json)
        df["llm_tldr"]         = parsed.map(lambda d: (d or {}).get("tldr"))
        df["llm_key_findings"] = parsed.map(lambda d: " | ".join((d or {}).get("key_findings") or []))
        df["llm_methods"]      = parsed.map(lambda d: (d or {}).get("methods"))
        df["llm_caveats"]      = parsed.map(lambda d: (d or {}).get("caveats"))

        n_ok = df["llm_tldr"].notna().sum()
        print(f"   Summaries produced: {n_ok}/{len(candidates)}")


# ============================================================================
# Notes that are NOT in the code, on purpose:
#
# 1. Sci-Hub fallback was suggested in the feedback. I deliberately did not
#    add it. Sci-Hub redistributes copyrighted papers without authorisation;
#    bundling it into a notebook you plan to publish would expose you (and
#    anyone forking the repo) to DMCA/takedown risk and would get the repo
#    flagged on GitHub. The legitimate stack above (PMC + Unpaywall +
#    Europe PMC + CrossRef + bioRxiv) is what actually gets you to the
#    60–70% yield range your reviewer mentioned, AS LONG AS USER_EMAIL is
#    set. If you want one more legitimate boost, add CORE (api.core.ac.uk,
#    free key) or Semantic Scholar's openAccessPdf field — both are
#    drop-in providers using the same `resolve(result)` contract.
#
# 2. Streamlit dashboard: that's a separate deliverable, not a Cell 7 fix.
#    The right move is a `streamlit_app.py` in the repo root that loads
#    the parquet/csv this notebook produces. Happy to write it as a
#    follow-up — it shouldn't live inside the scraper notebook.
#
# 3. OA Button: was a useful free aggregator but OA.Works shut it down on
#    2025-11-18. The domain still exists for the website redirect but the
#    API DNS record is gone, hence the NameResolutionErrors. Removed in v3.4.
# ============================================================================

In [ ]:
# ============================================================================
# CELL 7 — OA Full-Text Resolution v3.5.4 MAX  (clean, consolidated, drop-in)
#
# Replaces the v2 Unpaywall-only cell AND every "Cell 7 v3.x" you've pasted
# below it. Delete those — this is the only one you need.
#
# Resolution order (5 providers + 1 title-search fallback):
#   PMC → Unpaywall → Europe PMC → CrossRef → Semantic Scholar → bioRxiv/medRxiv
#   → (preprint title-similarity search if still no full text)
#
# What's new in 3.5 vs 3.4:
#   - NEW provider: SemanticScholarResolver (openAccessPdf field, free, no key)
#     — catches author-hosted PDFs that Unpaywall/CrossRef miss
#   - DEFAULTS flipped to MAX mode:
#       EXTRACT_PDF_TEXT      = True   (downloads + parses PDFs with pypdf)
#       PREPRINT_TITLE_SEARCH = True   (finds preprint twins via Jaccard)
#       PDF_TEXT_MAX_RECORDS  = 100    (was 20)
#       PDF_TEXT_MAX_BYTES    = 12 MB  (was 8 MB)
#   - Coverage report now also shows extracted-content stats, not just links
#
# Expected for a 50-record biomedical query with valid email:
#   - Full-text yield: 65-80% (vs 42% PMC-only)
#   - Extracted text:  ~90% of full-text URLs (some publisher PDFs gate scraping)
#   - Wall time: ~60-90s (PDF downloads dominate; tune PDF_TEXT_MAX_RECORDS down
#     if you want it faster and don't need the actual text)
# ============================================================================
import os, sys, time, logging, re

# ---- 0. Config pulled from earlier cells, with explicit precedence ---------
# Hardcoded so this works even if Cell 2 wasn't re-run / email got reverted.
# If you change your email later, edit this line directly.
USER_EMAIL    = "abrahamtrueba@berkeley.edu"
# (Falls back to globals if you ever blank the line above)
USER_EMAIL    = USER_EMAIL or (globals().get("USER_EMAIL") or globals().get("NCBI_EMAIL") or "").strip()
NCBI_API_KEY  = globals().get("NCBI_API_KEY") or None
ENRICH_UNPAYWALL = bool(globals().get("ENRICH_UNPAYWALL", True))

# v3.5.4 MAX-COVERAGE MODE — these are HARDCODED, ignoring stale globals from
# previous Cell 2 runs. To customize, edit these lines directly.
EXTRACT_PDF_TEXT       = True             # download + parse OA PDFs with pypdf
PDF_TEXT_MAX_BYTES     = 12_000_000       # 12 MB cap per PDF
PDF_TEXT_MAX_RECORDS   = 100              # process up to this many in a batch
PREPRINT_TITLE_SEARCH  = True             # bioRxiv/medRxiv title-similarity fallback
GROQ_SUMMARIES         = bool(globals().get("GROQ_SUMMARIES", False))   # OFF — needs GROQ_API_KEY
GROQ_MODEL             = globals().get("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_MAX_RECORDS       = int(globals().get("GROQ_MAX_RECORDS", 30))

_PLACEHOLDER_EMAILS = {"", "your@email.com", "research@example.com",
                       "you@yourdomain.com", "user@example.com"}
_email_is_real = USER_EMAIL.lower() not in _PLACEHOLDER_EMAILS and "@" in USER_EMAIL

if not _email_is_real:
    print("=" * 72)
    print("  ⚠️  USER_EMAIL IS UNSET OR A PLACEHOLDER")
    print("  ----------------------------------------")
    print("  Unpaywall and CrossRef require a real email per their ToS.")
    print("  Without it, OA yield drops to PMC-only (~40-45% on biomedical sets).")
    print("  With a real email, expect 60-70% on broad PubMed queries.")
    print("")
    print("  Fix: in Cell 2, set")
    print("       NCBI_EMAIL = 'you@your.domain'")
    print("  then re-run this cell. No other changes needed.")
    print("=" * 72)
    print()

# ---- 1. Resolver module (written ONCE) -------------------------------------
_MODULE_PATH = "/content/oa_fulltext_v3.py"
_MODULE_SRC = r'''
from __future__ import annotations
import logging, re, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

log = logging.getLogger("bioscraper.oa_v3")


def _build_session(user_agent="NCBI-BioScraper-Pro/3.5"):
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


def _build_session_no_429_retry(user_agent="NCBI-BioScraper-Pro/3.5"):
    """Like _build_session but does NOT auto-retry 429s. Used for providers
    where we want to see the 429 status code directly (e.g. so a self-throttle
    latch can engage instead of letting urllib3 burn 3 retries silently)."""
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=(500, 502, 503, 504),  # 429 deliberately omitted
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


@dataclass
class OAResult:
    pmid: Optional[str] = None
    doi: Optional[str] = None
    pmcid: Optional[str] = None
    oa_source: str = "none"
    oa_pdf_url: Optional[str] = None
    oa_xml_url: Optional[str] = None
    oa_html_url: Optional[str] = None
    oa_license: Optional[str] = None
    oa_status: str = "closed"
    oa_full_text: bool = False
    oa_attempts: list = field(default_factory=list)
    oa_resolved_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def as_flat_dict(self):
        d = asdict(self)
        d["oa_attempts"] = ",".join(d["oa_attempts"])
        return d


# ---------- providers --------------------------------------------------------
class PMCResolver:
    IDCONV = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    OA_FCGI = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"

    def __init__(self, session, ncbi_api_key=None, tool="ncbi-bioscraper", email="research@example.com"):
        self.s = session; self.api_key = ncbi_api_key; self.tool = tool; self.email = email

    def _params(self, **extra):
        p = {"tool": self.tool, "email": self.email}
        if self.api_key: p["api_key"] = self.api_key
        p.update(extra); return p

    def pmid_to_pmcid(self, pmid, doi=None):
        for key, value in (("pmid", pmid), ("doi", doi)):
            if not value: continue
            try:
                r = self.s.get(self.IDCONV,
                               params=self._params(ids=value, idtype=key, format="json"),
                               timeout=6)
                if r.status_code != 200: continue
                records = r.json().get("records", [])
                if records and records[0].get("pmcid"):
                    return records[0]["pmcid"]
            except Exception as e:
                log.debug("idconv (%s=%s) failed: %s", key, value, e)
        return None

    def pmcid_to_oa(self, pmcid):
        try:
            r = self.s.get(self.OA_FCGI, params={"id": pmcid}, timeout=6)
            if r.status_code != 200 or "<error" in r.text.lower():
                return None
            text = r.text
            pdf_url = xml_url = license_str = None
            for line in text.split("<link "):
                if 'format="pdf"' in line and 'href="' in line:
                    pdf_url = line.split('href="')[1].split('"')[0]
                elif 'format="tgz"' in line and 'href="' in line:
                    xml_url = line.split('href="')[1].split('"')[0]
            if 'license="' in text:
                license_str = text.split('license="')[1].split('"')[0]
            if pdf_url or xml_url:
                if pdf_url and pdf_url.startswith("ftp://"):
                    pdf_url = "https://" + pdf_url[len("ftp://"):]
                if xml_url and xml_url.startswith("ftp://"):
                    xml_url = "https://" + xml_url[len("ftp://"):]
                return {"pdf_url": pdf_url, "xml_url": xml_url,
                        "license": license_str,
                        "html_url": "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % pmcid}
        except Exception as e:
            log.debug("oa.fcgi failed for %s: %s", pmcid, e)
        return None

    def resolve(self, result):
        result.oa_attempts.append("pmc")
        if not result.pmcid:
            result.pmcid = self.pmid_to_pmcid(result.pmid, result.doi)
        if not result.pmcid:
            return False
        oa = self.pmcid_to_oa(result.pmcid)
        if not oa:
            # PMC-hosted but not in OA subset — still useful as a reader URL
            result.oa_html_url = "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % result.pmcid
            return False
        result.oa_source = "pmc"
        result.oa_pdf_url = oa.get("pdf_url")
        result.oa_xml_url = oa.get("xml_url")
        result.oa_html_url = oa.get("html_url")
        result.oa_license = oa.get("license")
        result.oa_status = "gold"
        result.oa_full_text = bool(result.oa_pdf_url or result.oa_xml_url)
        return result.oa_full_text


class UnpaywallResolver:
    BASE = "https://api.unpaywall.org/v2/"

    def __init__(self, session, email):
        self.s = session; self.email = email
        self._not_in_index = 0
        self._auth_failed = False  # latched: stop calling once we see a 422

    def resolve(self, result):
        result.oa_attempts.append("unpaywall")
        if self._auth_failed or not result.doi:
            return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"email": self.email}, timeout=6)
            if r.status_code == 404:
                self._not_in_index += 1
                return False
            if r.status_code == 422:
                # Auth/email problem — latch and stop hammering the API
                self._auth_failed = True
                log.warning("Unpaywall 422 — disabling Unpaywall for the rest of this run "
                            "(check USER_EMAIL is a real address).")
                return False
            if r.status_code != 200:
                return False
            d = r.json()
            if not d.get("is_oa"):
                return False
            best = d.get("best_oa_location") or {}
            if not best:
                locs = d.get("oa_locations") or []
                if locs: best = locs[0]
            if not best: return False
            pdf  = best.get("url_for_pdf")
            html = best.get("url_for_landing_page") or best.get("url")
            if not (pdf or html): return False
            result.oa_source = "unpaywall"
            result.oa_pdf_url = pdf
            result.oa_html_url = html
            result.oa_license = best.get("license")
            result.oa_status = d.get("oa_status") or "bronze"
            result.oa_full_text = bool(pdf)
            return bool(pdf or html)
        except Exception as e:
            log.debug("unpaywall failed for %s: %s", result.doi, e)
            return False


class EuropePMCResolver:
    SEARCH = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def __init__(self, session): self.s = session

    def resolve(self, result):
        result.oa_attempts.append("europepmc")
        if result.pmid:
            query = "EXT_ID:%s AND SRC:MED" % result.pmid
        elif result.doi:
            query = 'DOI:"%s"' % result.doi
        else:
            return False
        try:
            r = self.s.get(self.SEARCH, params={
                "query": query, "format": "json",
                "resultType": "core", "pageSize": 1,
            }, timeout=6)
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            if not hits: return False
            h = hits[0]
            if h.get("isOpenAccess") != "Y": return False
            full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
            pdf = next((u["url"] for u in full_text_urls
                        if u.get("documentStyle", "").lower() == "pdf"
                        and u.get("availability", "").lower() == "open access"), None)
            html = next((u["url"] for u in full_text_urls
                         if u.get("documentStyle", "").lower() == "html"), None)
            if not (pdf or html): return False
            result.oa_source = "europepmc"
            result.oa_pdf_url = pdf or result.oa_pdf_url
            result.oa_html_url = html or result.oa_html_url
            result.oa_license = h.get("license") or result.oa_license
            result.oa_status = "gold" if h.get("inEPMC") == "Y" else "green"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("europepmc failed: %s", e)
            return False


class CrossRefOAResolver:
    """Catches recent papers (<30 days) that Unpaywall hasn't ingested yet,
    and any paper with a CC license declared in CrossRef metadata."""
    BASE = "https://api.crossref.org/works/"

    def __init__(self, session, email):
        self.s = session; self.email = email

    def resolve(self, result):
        result.oa_attempts.append("crossref")
        if not result.doi: return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"mailto": self.email}, timeout=6)
            if r.status_code != 200: return False
            msg = r.json().get("message") or {}
            licenses = msg.get("license") or []
            cc = next((L for L in licenses
                       if "creativecommons.org" in (L.get("URL") or "").lower()), None)
            if not cc: return False
            license_url = cc.get("URL", "")
            license_id = None
            if   "/by-nc-nd" in license_url: license_id = "cc-by-nc-nd"
            elif "/by-nc-sa" in license_url: license_id = "cc-by-nc-sa"
            elif "/by-nc"    in license_url: license_id = "cc-by-nc"
            elif "/by-sa"    in license_url: license_id = "cc-by-sa"
            elif "/by"       in license_url: license_id = "cc-by"
            elif "/zero"     in license_url: license_id = "cc0"
            pdf = None
            for link in (msg.get("link") or []):
                if (link.get("content-type") == "application/pdf"
                        and "vor" in (link.get("intended-application", "") or "").lower()):
                    pdf = link.get("URL"); break
            result.oa_source = "crossref"
            result.oa_pdf_url = pdf
            result.oa_html_url = "https://doi.org/" + result.doi
            result.oa_license = license_id or "cc-detected"
            result.oa_status = "gold"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("crossref failed for %s: %s", result.doi, e)
            return False


class SemanticScholarResolver:
    """Semantic Scholar's openAccessPdf field. Free, no key needed at low
    volume (~1 req/sec unauthenticated). Catches a different slice than
    Unpaywall/CrossRef — especially newer papers and CS/biology preprints
    where their crawler picks up author-hosted PDFs.

    Self-rate-limits to 1 req/sec to avoid 429 cascades.
    Skips itself entirely if `result.oa_full_text` is already True from an
    earlier provider — saves quota for records that actually need it.
    Uses its own session that does NOT retry 429s, so the latch can engage
    on the first 429 instead of letting urllib3 burn 3 silent retries.
    """
    BASE = "https://api.semanticscholar.org/graph/v1/paper/"
    MIN_INTERVAL = 1.05  # seconds between calls

    def __init__(self, session=None, email=None):
        # Ignore the shared session — S2 needs the no-retry-on-429 behaviour
        self.s = _build_session_no_429_retry()
        self.email = email
        self._last_call = 0.0
        self._rate_limited = False  # latch after a 429 to stop hammering

    def _lookup(self, identifier):
        if self._rate_limited:
            return None
        # Self-throttle to 1 req/sec
        elapsed = time.time() - self._last_call
        if elapsed < self.MIN_INTERVAL:
            time.sleep(self.MIN_INTERVAL - elapsed)
        self._last_call = time.time()
        try:
            r = self.s.get(self.BASE + identifier,
                           params={"fields": "openAccessPdf,externalIds,isOpenAccess,title"},
                           timeout=6)
            if r.status_code == 429:
                # Latch off — S2 has decided we've had enough. Don't keep retrying.
                self._rate_limited = True
                log.warning("Semantic Scholar 429 — disabling S2 for the rest of this run.")
                return None
            if r.status_code != 200:
                return None
            return r.json()
        except Exception as e:
            # If urllib3 still managed to hit retry exhaustion on 429s
            # (shouldn't with our no-retry session, but belt-and-braces),
            # treat that as a hard latch too.
            err_str = str(e).lower()
            if "429" in err_str or "too many" in err_str:
                self._rate_limited = True
                log.warning("Semantic Scholar repeated 429 — disabling S2 for the rest of this run.")
            else:
                log.debug("semanticscholar lookup %s failed: %s", identifier, e)
            return None

    def resolve(self, result):
        # Skip entirely if an earlier provider already gave us a PDF
        if result.oa_full_text:
            return False
        result.oa_attempts.append("semanticscholar")
        d = None
        if result.doi:
            d = self._lookup("DOI:" + result.doi)
        if not d and result.pmid:
            d = self._lookup("PMID:" + result.pmid)
        if not d:
            return False
        oa_pdf = d.get("openAccessPdf") or {}
        url = oa_pdf.get("url")
        if not url:
            return False
        result.oa_source = "semanticscholar"
        result.oa_pdf_url = url
        result.oa_html_url = result.oa_html_url or url
        result.oa_license = oa_pdf.get("license") or result.oa_license or "s2-detected"
        if result.oa_status == "closed":
            result.oa_status = "green"
        result.oa_full_text = True
        return True


# OAButtonResolver removed in v3.4: OA.Works retired the openaccessbutton.org
# API on 2025-11-18. Calls were 100% failing with DNS errors. If you want a
# similar drop-in replacement, CORE (api.core.ac.uk, free key) or Semantic
# Scholar's openAccessPdf field both work and follow the same `resolve(result)`
# contract.


class PreprintResolver:
    """Direct lookup for 10.1101/* DOIs (bioRxiv/medRxiv)."""
    def __init__(self, session): self.s = session


    def resolve(self, result):
        result.oa_attempts.append("preprint")
        if not result.doi or not result.doi.startswith("10.1101/"):
            return False
        for host in ("www.biorxiv.org", "www.medrxiv.org"):
            url = "https://%s/content/%s.full.pdf" % (host, result.doi)
            try:
                r = self.s.head(url, timeout=8, allow_redirects=True)
                if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
                    result.oa_source = "preprint"
                    result.oa_pdf_url = url
                    result.oa_html_url = url.rsplit(".full.pdf", 1)[0] + ".full"
                    result.oa_status = "green"
                    result.oa_full_text = True
                    return True
            except Exception:
                continue
        return False


class PreprintTitleSearchResolver:
    """Last-resort: search bioRxiv/medRxiv by title and accept matches with
    high token-overlap similarity. Useful when a peer-reviewed paper has
    a preprint twin under a different DOI."""
    BIORXIV_API = "https://api.biorxiv.org/details/{server}/{date_or_doi}/0/json"
    SEARCH = "https://www.biorxiv.org/search/{query}"  # HTML, not used directly

    def __init__(self, session, min_similarity=0.75):
        self.s = session
        self.min_similarity = min_similarity

    @staticmethod
    def _norm_tokens(s):
        s = (s or "").lower()
        s = re.sub(r"[^a-z0-9 ]+", " ", s)
        return set(t for t in s.split() if len(t) > 2)

    def _similarity(self, a, b):
        ta, tb = self._norm_tokens(a), self._norm_tokens(b)
        if not ta or not tb: return 0.0
        return len(ta & tb) / len(ta | tb)  # Jaccard

    def resolve(self, result, candidate_title=None):
        result.oa_attempts.append("preprint_search")
        if not candidate_title: return False
        # bioRxiv has no native title-search API; we use Europe PMC restricted
        # to preprint sources, which DOES index bioRxiv/medRxiv records.
        try:
            r = self.s.get(
                "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                params={"query": 'TITLE:"%s" AND (SRC:PPR)' % candidate_title[:200],
                        "format": "json", "resultType": "core", "pageSize": 5},
                timeout=6,
            )
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            for h in hits:
                sim = self._similarity(candidate_title, h.get("title"))
                if sim < self.min_similarity: continue
                doi = h.get("doi")
                full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
                pdf = next((u["url"] for u in full_text_urls
                            if u.get("documentStyle", "").lower() == "pdf"), None)
                html = next((u["url"] for u in full_text_urls
                             if u.get("documentStyle", "").lower() == "html"), None)
                if not (pdf or html): continue
                result.oa_source = "preprint_search"
                result.oa_pdf_url = pdf or result.oa_pdf_url
                result.oa_html_url = html or result.oa_html_url
                result.oa_status = "green"
                result.oa_full_text = bool(pdf)
                return True
        except Exception as e:
            log.debug("preprint_search failed: %s", e)
        return False


# ---------- orchestrator -----------------------------------------------------
class OAFullTextResolver:
    def __init__(self, email, ncbi_api_key=None,
                 enable_pmc=True, enable_unpaywall=True,
                 enable_europepmc=True, enable_crossref=True,
                 enable_semanticscholar=True,
                 enable_oabutton=False,  # kept for back-compat; ignored (API retired)
                 enable_preprint=True,
                 enable_preprint_search=False,
                 polite_delay_s=0.0):
        self.session = _build_session()
        self.polite_delay = polite_delay_s
        self.providers = []
        if enable_pmc:
            self.providers.append(PMCResolver(self.session, ncbi_api_key, email=email))
        if enable_unpaywall:
            self.providers.append(UnpaywallResolver(self.session, email=email))
        if enable_europepmc:
            self.providers.append(EuropePMCResolver(self.session))
        if enable_crossref:
            self.providers.append(CrossRefOAResolver(self.session, email=email))
        if enable_semanticscholar:
            self.providers.append(SemanticScholarResolver(self.session, email=email))
        # OA Button removed in v3.4 (api.openaccessbutton.org retired 2025-11-18)
        if enable_preprint:
            self.providers.append(PreprintResolver(self.session))
        # Title-search resolver is held separately because it needs a title arg
        self.preprint_search = (PreprintTitleSearchResolver(self.session)
                                if enable_preprint_search else None)

    def resolve_one(self, pmid=None, doi=None, pmcid=None, title=None):
        result = OAResult(pmid=pmid, doi=doi, pmcid=pmcid)
        for provider in self.providers:
            try:
                if provider.resolve(result) and result.oa_full_text:
                    break
            except Exception as e:
                log.warning("%s raised: %s", type(provider).__name__, e)
            if self.polite_delay:
                time.sleep(self.polite_delay)
        # Title-similarity fallback only if we still don't have full text
        if (self.preprint_search is not None
                and not result.oa_full_text
                and title):
            try:
                self.preprint_search.resolve(result, candidate_title=title)
            except Exception as e:
                log.warning("PreprintTitleSearchResolver raised: %s", e)
        return result

    def enrich_dataframe(self, df, pmid_col="pmid", doi_col="doi",
                         pmcid_col=None, title_col=None, show_progress=True):
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(df.iterrows(), total=len(df),
                            desc="OA resolve", disable=not show_progress)
        except ImportError:
            iterator = df.iterrows()

        rows = []
        import pandas as pd
        for _, row in iterator:
            def _get(col):
                if col and col in df.columns:
                    v = row[col]
                    if v is None: return None
                    try:
                        if pd.isna(v): return None
                    except Exception: pass
                    s = str(v).strip()
                    return s if s and s.lower() != "nan" else None
                return None
            rows.append(self.resolve_one(
                pmid=_get(pmid_col),
                doi=_get(doi_col),
                pmcid=_get(pmcid_col),
                title=_get(title_col),
            ).as_flat_dict())

        oa_df = pd.DataFrame(rows)
        # Don't re-emit identifier cols already present on the input df, otherwise
        # a second pass through enrich_dataframe will collide on pmid/doi/pmcid.
        for c in ("pmid", "doi", "pmcid"):
            if c in oa_df.columns and c in df.columns:
                oa_df = oa_df.drop(columns=[c])
        for c in list(df.columns):
            if c.startswith("oa_") or c.startswith("up_"):
                df = df.drop(columns=[c])
        return df.reset_index(drop=True).join(oa_df.reset_index(drop=True))

    def coverage_report(self, df):
        total = len(df)
        if total == 0: return {"total": 0}
        return {
            "total": total,
            "full_text_pct": round(100 * df["oa_full_text"].sum() / total, 1),
            "any_oa_link_pct": round(100 * (df["oa_source"] != "none").sum() / total, 1),
            "by_source": df["oa_source"].value_counts().to_dict(),
            "by_status": df["oa_status"].value_counts().to_dict(),
        }
'''

# Write it ONCE, force a clean import
with open(_MODULE_PATH, "w", encoding="utf-8") as f:
    f.write(_MODULE_SRC)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
for _mod in [m for m in sys.modules if m.startswith("oa_fulltext_v3")]:
    del sys.modules[_mod]

from oa_fulltext_v3 import OAFullTextResolver, UnpaywallResolver  # noqa: E402

# ---- 2. Build resolver with sane provider gating ---------------------------
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("bioscraper.oa_v3").setLevel(logging.WARNING)  # quiet the per-record DEBUG noise

oa = OAFullTextResolver(
    email=USER_EMAIL or "research@example.com",
    ncbi_api_key=NCBI_API_KEY,
    enable_pmc=True,
    enable_unpaywall=ENRICH_UNPAYWALL and _email_is_real,   # << the fix
    enable_europepmc=True,
    enable_crossref=_email_is_real,                          # CrossRef wants mailto too
    # OA Button retired 2025-11-18, kwarg dropped
    enable_preprint=True,
    enable_preprint_search=PREPRINT_TITLE_SEARCH,
)

# ---- 3. Enrich -------------------------------------------------------------
# Self-heal: if `df` was knocked out of the namespace by a prior error or
# kernel reset, rebuild it from the SQLite cache from earlier scrapes.
def _rebuild_df_from_cache():
    """Rebuild a minimal df (pmid/doi/title) from the pubmed_cache SQLite table.
    Only enough columns for Cell 7 to operate. Does not re-fetch from PubMed."""
    import sqlite3, json as _json
    import pandas as _pd

    candidate_paths = []
    if "CACHE_DB" in globals():
        candidate_paths.append(globals()["CACHE_DB"])
    if "OUTPUT_DIR" in globals():
        candidate_paths.append(os.path.join(globals()["OUTPUT_DIR"], "_cache.sqlite"))
    # Common defaults if config globals also got wiped
    candidate_paths += [
        "/content/output/_cache.sqlite",
        "/content/drive/MyDrive/NCBI_Research/crispr_cancer_v2/_cache.sqlite",
    ]

    db_path = next((p for p in candidate_paths if p and os.path.exists(p)), None)
    if not db_path:
        return None, "no cache file found at: " + ", ".join(str(p) for p in candidate_paths)

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.execute("SELECT pmid, record_json FROM pubmed_cache")
        rows = []
        for pmid, blob in cursor.fetchall():
            try:
                rec = _json.loads(blob) if blob else {}
            except Exception:
                rec = {}
            doi = (rec.get("LID") or rec.get("AID") or "")
            if isinstance(doi, list): doi = doi[0] if doi else ""
            doi = str(doi).strip()
            # Strip trailing "[doi]" tag PubMed sometimes attaches
            doi = re.sub(r"\s*\[doi\]\s*$", "", doi, flags=re.IGNORECASE).strip()
            # Keep only DOIs that look like DOIs (start with 10.)
            if doi and not doi.startswith("10."):
                m = re.search(r"\b(10\.\d{4,9}/[^\s]+)", doi)
                doi = m.group(1) if m else ""
            rows.append({
                "pmid":  str(pmid),
                "doi":   doi,
                "title": str(rec.get("TI", "")),
                "abstract": str(rec.get("AB", "")),
                "pmcid": str(rec.get("PMC", "") or ""),
            })
        conn.close()
        if not rows:
            return None, f"cache at {db_path} has 0 records"
        return _pd.DataFrame(rows), f"loaded {len(rows)} records from {db_path}"
    except Exception as e:
        return None, f"cache read failed at {db_path}: {e}"


if "df" not in globals() or globals().get("df") is None:
    print("⚠️  `df` not in scope — rebuilding from SQLite cache…")
    import pandas as pd
    rebuilt, msg = _rebuild_df_from_cache()
    print(f"   {msg}")
    if rebuilt is None or len(rebuilt) == 0:
        print("\n❌ Could not rebuild `df`. Run Cell 5 (PubMed scrape) first,")
        print("   then re-run this cell.")
        raise RuntimeError("df missing and cache empty/unreadable")
    df = rebuilt
    print(f"   ✅ Rebuilt df with columns: {list(df.columns)}")
    print(f"   ✅ {len(df)} records ready for OA resolution\n")

# Use 'title' column if it exists, for the preprint title-similarity fallback
_title_col = "title" if "title" in df.columns else None
df = df.drop(columns=[c for c in df.columns if c.startswith("oa_") or c.startswith("up_")],
             errors="ignore")
df = oa.enrich_dataframe(df, pmid_col="pmid", doi_col="doi", title_col=_title_col)

# ---- 4. Coverage report ----------------------------------------------------
report = oa.coverage_report(df)
print("\n📊 OA full-text resolution v3.5.4 MAX (%d records)" % report["total"])
print("   Full-text yield: %s%%" % report["full_text_pct"])
print("   Any OA link:     %s%%" % report["any_oa_link_pct"])
print("   By source:       %s" % report["by_source"])
print("   By status:       %s" % report["by_status"])

unpaywall_obj = next((p for p in oa.providers if isinstance(p, UnpaywallResolver)), None)
if unpaywall_obj is not None:
    n_miss = unpaywall_obj._not_in_index
    print(f"\n   Unpaywall index misses (404s): {n_miss}/{len(df)}")
    print( "   -> these are typically <30-day-old DOIs not yet ingested by Unpaywall")
    if unpaywall_obj._auth_failed:
        print("   ⚠️  Unpaywall auth failed mid-run — fix USER_EMAIL and re-run this cell.")
elif ENRICH_UNPAYWALL and not _email_is_real:
    print("\n   Unpaywall: SKIPPED (placeholder email).")


# ============================================================================
# CELL 7b — Download PDFs and extract plain text
# Controlled by EXTRACT_PDF_TEXT flag. ON by default in v3.5 MAX.
# Uses pypdf only (no GPU, no OCR). Caps at PDF_TEXT_MAX_RECORDS for safety.
# ============================================================================
print()
print("─" * 72)
print(f"  PDF text extraction: {'ON' if EXTRACT_PDF_TEXT else 'OFF'}  "
      f"(EXTRACT_PDF_TEXT = {EXTRACT_PDF_TEXT})")
if EXTRACT_PDF_TEXT:
    print(f"  Cap: {PDF_TEXT_MAX_RECORDS} PDFs, "
          f"{PDF_TEXT_MAX_BYTES // 1_000_000} MB each")
print("─" * 72)

if EXTRACT_PDF_TEXT:
    try:
        import pypdf  # noqa: F401
    except ImportError:
        print("Installing pypdf for text extraction...")
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
        import pypdf

    import io, requests as _rq
    from collections import Counter as _Counter
    from pypdf import PdfReader

    # Browser-style User-Agent — many publishers block obvious scrapers but
    # serve OA content fine to a normal-looking browser fetch. This is OA
    # content the publisher has affirmatively licensed for free reading.
    _PDF_UA = ("Mozilla/5.0 (X11; Linux x86_64) "
               "AppleWebKit/537.36 (KHTML, like Gecko) "
               "Chrome/124.0.0.0 Safari/537.36")
    _PDF_HEADERS = {
        "User-Agent": _PDF_UA,
        "Accept": "application/pdf,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    # Track precise failure reasons so we can SEE why extraction is failing
    # rather than guessing. This is the key change in v3.5.3.
    _failure_reasons = []   # list of (idx, source, url, reason, detail)

    def _extract_pdf_text(url, max_bytes=PDF_TEXT_MAX_BYTES, idx=None, source=None):
        if not url:
            _failure_reasons.append((idx, source, url, "no_url", ""))
            return None
        try:
            with _rq.get(url, stream=True, timeout=20,
                         headers=_PDF_HEADERS, allow_redirects=True) as r:
                if r.status_code != 200:
                    _failure_reasons.append(
                        (idx, source, url, f"http_{r.status_code}",
                         f"final_url={r.url}"))
                    return None
                ctype = (r.headers.get("content-type") or "").lower()
                final_url = r.url
                if "pdf" not in ctype and not final_url.lower().endswith(".pdf"):
                    # Likely redirected to a login wall or HTML reader page
                    _failure_reasons.append(
                        (idx, source, url, "not_pdf", f"ctype={ctype} final={final_url}"))
                    return None
                buf = io.BytesIO()
                total = 0
                for chunk in r.iter_content(chunk_size=65536):
                    if not chunk: continue
                    buf.write(chunk); total += len(chunk)
                    if total > max_bytes:
                        _failure_reasons.append(
                            (idx, source, url, "too_big", f"{total:,} bytes"))
                        return None
                buf.seek(0)
                try:
                    reader = PdfReader(buf)
                except Exception as e:
                    _failure_reasons.append(
                        (idx, source, url, "pdf_parse_error", str(e)[:120]))
                    return None
                pages = []
                for page in reader.pages:
                    try: pages.append(page.extract_text() or "")
                    except Exception: pass
                text = "\n\n".join(pages).strip()
                text = re.sub(r"[ \t]+", " ", text)
                text = re.sub(r"\n{3,}", "\n\n", text)
                if not text:
                    _failure_reasons.append(
                        (idx, source, url, "empty_text", f"{total:,} bytes downloaded"))
                    return None
                return text
        except _rq.exceptions.Timeout:
            _failure_reasons.append((idx, source, url, "timeout", ""))
        except _rq.exceptions.SSLError as e:
            _failure_reasons.append((idx, source, url, "ssl_error", str(e)[:120]))
        except Exception as e:
            _failure_reasons.append((idx, source, url, "exception", str(e)[:120]))
        return None

    eligible = df[df["oa_full_text"] & df["oa_pdf_url"].notna()].copy()
    priority_order = ["pmc", "preprint", "preprint_search", "europepmc",
                      "unpaywall", "semanticscholar", "crossref"]
    eligible["_prio"] = eligible["oa_source"].map(
        {s: i for i, s in enumerate(priority_order)}).fillna(99)
    eligible = eligible.sort_values("_prio").head(PDF_TEXT_MAX_RECORDS)

    # Diagnostic context: report what we're processing and what we're skipping
    n_full_text  = int(df["oa_full_text"].sum())
    n_with_pdf   = int(df["oa_pdf_url"].notna().sum())
    n_xml_only   = int((df["oa_full_text"] & df["oa_pdf_url"].isna() &
                        df["oa_xml_url"].notna()).sum())

    if len(eligible) == 0:
        print("\n  ⚠️  No records with full-text PDF URLs to extract.")
        print("     (Need oa_full_text=True AND oa_pdf_url not null.)")
        df["oa_full_text_extracted"] = None
        df["oa_full_text_chars"] = 0
    else:
        print(f"\n📄 Extraction queue ({len(eligible)} of {n_full_text} full-text records):")
        print(f"   Records flagged full_text=True:    {n_full_text}")
        print(f"   Records with a PDF URL:            {n_with_pdf}")
        if n_xml_only:
            print(f"   Records with XML/tgz only (skipped): {n_xml_only}")
        print(f"   By source in queue: {dict(eligible['oa_source'].value_counts())}")

        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(eligible.iterrows(), total=len(eligible), desc="PDF extract")
        except ImportError:
            it = eligible.iterrows()

        texts = {}
        for idx, row in it:
            t = _extract_pdf_text(row["oa_pdf_url"], idx=idx, source=row["oa_source"])
            if t: texts[idx] = t

        df["oa_full_text_extracted"] = df.index.map(texts).where(df.index.isin(texts), None)
        df["oa_full_text_chars"]     = df["oa_full_text_extracted"].fillna("").str.len()
        n_extracted = int(df["oa_full_text_chars"].gt(0).sum())
        median_chars = int(df.loc[df["oa_full_text_chars"] > 0, "oa_full_text_chars"].median() or 0)
        print(f"\n   Extracted: {n_extracted}/{len(eligible)} PDFs successfully parsed")
        if n_extracted:
            print(f"   Median length: {median_chars:,} chars (~{median_chars//5_000} pages of plain text)")
            print(f"   Total content captured: {int(df['oa_full_text_chars'].sum()):,} chars")

        # Per-source success rate — tells us which providers gave usable PDFs
        succ_by_src = df.loc[df["oa_full_text_chars"] > 0, "oa_source"].value_counts().to_dict()
        attempted_by_src = eligible["oa_source"].value_counts().to_dict()
        print(f"\n   Per-source extraction success:")
        for src in attempted_by_src:
            ok = succ_by_src.get(src, 0); tot = attempted_by_src[src]
            print(f"     {src:18s} {ok}/{tot}")

        # Failure-reason breakdown — the diagnostic that was missing
        if _failure_reasons:
            reason_counts = _Counter(fr[3] for fr in _failure_reasons)
            print(f"\n   Failure reasons ({len(_failure_reasons)} total):")
            for reason, count in reason_counts.most_common():
                print(f"     {reason:18s} {count}")
            # Show first few examples per reason category
            print(f"\n   Failure examples (up to 5):")
            for fr in _failure_reasons[:5]:
                idx_, src_, url_, reason_, detail_ = fr
                short_url = url_[:60] + "…" if url_ and len(url_) > 60 else (url_ or "")
                print(f"     [{src_}] {reason_}: {short_url}")
                if detail_: print(f"       └─ {detail_[:140]}")

        # Stash the failure log on df for post-hoc inspection
        df.attrs["pdf_extract_failures"] = _failure_reasons

        print(f"\n   Records with ABSTRACT only:        {len(df) - n_extracted}")
        print(f"   Records with FULL TEXT extracted:  {n_extracted}")
        print(f"   -> use df['oa_full_text_extracted'] for downstream NLP / LLM input")
        print(f"   -> use df.attrs['pdf_extract_failures'] to inspect failed downloads")


# ============================================================================
# CELL 7c — Optional: Groq LLM summaries + key findings
# Off by default. Requires:  pip install groq   AND   os.environ["GROQ_API_KEY"]
# Operates on EXTRACTED full text if available, else on the abstract.
# ============================================================================
if GROQ_SUMMARIES:
    api_key = os.environ.get("GROQ_API_KEY", "").strip()
    if not api_key:
        print("\n⚠️  GROQ_SUMMARIES=True but GROQ_API_KEY not set in environment. Skipping.")
    else:
        try:
            from groq import Groq  # noqa: F401
        except ImportError:
            print("Installing groq client…")
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
        from groq import Groq

        client = Groq(api_key=api_key)

        SYSTEM_PROMPT = (
            "You are summarising a biomedical paper for a domain expert in "
            "molecular biology, genomics, and computational biology. "
            "Return STRICT JSON with exactly these keys: "
            '{"tldr": str (<=40 words), '
            '"key_findings": list[str] (3-5 bullets, each <=25 words), '
            '"methods": str (<=30 words), '
            '"caveats": str (<=25 words, or empty string if none)}. '
            "Stay concrete. Cite figures or numbers when they appear in the source. "
            "Do not invent details. Output ONLY the JSON object."
        )

        def _summarise_one(text, model=GROQ_MODEL, max_chars=24000):
            text = (text or "").strip()
            if not text: return None
            payload = text[:max_chars]
            try:
                resp = client.chat.completions.create(
                    model=model,
                    temperature=0.2,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": payload},
                    ],
                )
                return resp.choices[0].message.content
            except Exception as e:
                logging.getLogger("bioscraper.oa_v3").warning("Groq summarise failed: %s", e)
                return None

        # Choose source text: extracted full text > abstract
        has_fulltext = ("oa_full_text_extracted" in df.columns)
        candidates = df.copy()
        if has_fulltext:
            candidates["_src"] = candidates["oa_full_text_extracted"].fillna(
                candidates.get("abstract", ""))
        else:
            candidates["_src"] = candidates.get("abstract", "")
        candidates = candidates[candidates["_src"].fillna("").str.len() > 200]
        candidates = candidates.head(GROQ_MAX_RECORDS)

        print(f"\n🤖 Summarising {len(candidates)} records via Groq ({GROQ_MODEL})…")
        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(candidates.iterrows(), total=len(candidates), desc="LLM summarise")
        except ImportError:
            it = candidates.iterrows()

        summaries = {}
        for idx, row in it:
            s = _summarise_one(row["_src"])
            if s: summaries[idx] = s

        df["llm_summary_json"] = df.index.map(summaries).where(df.index.isin(summaries), None)
        # Lightly parse for convenience
        import json as _json
        def _safe_json(s):
            try: return _json.loads(s) if s else None
            except Exception: return None
        parsed = df["llm_summary_json"].map(_safe_json)
        df["llm_tldr"]         = parsed.map(lambda d: (d or {}).get("tldr"))
        df["llm_key_findings"] = parsed.map(lambda d: " | ".join((d or {}).get("key_findings") or []))
        df["llm_methods"]      = parsed.map(lambda d: (d or {}).get("methods"))
        df["llm_caveats"]      = parsed.map(lambda d: (d or {}).get("caveats"))

        n_ok = df["llm_tldr"].notna().sum()
        print(f"   Summaries produced: {n_ok}/{len(candidates)}")


# ============================================================================
# Notes that are NOT in the code, on purpose:
#
# 1. Sci-Hub fallback was suggested in the feedback. I deliberately did not
#    add it. Sci-Hub redistributes copyrighted papers without authorisation;
#    bundling it into a notebook you plan to publish would expose you (and
#    anyone forking the repo) to DMCA/takedown risk and would get the repo
#    flagged on GitHub. The legitimate stack above (PMC + Unpaywall +
#    Europe PMC + CrossRef + bioRxiv) is what actually gets you to the
#    60–70% yield range your reviewer mentioned, AS LONG AS USER_EMAIL is
#    set. If you want one more legitimate boost, add CORE (api.core.ac.uk,
#    free key) or Semantic Scholar's openAccessPdf field — both are
#    drop-in providers using the same `resolve(result)` contract.
#
# 2. Streamlit dashboard: that's a separate deliverable, not a Cell 7 fix.
#    The right move is a `streamlit_app.py` in the repo root that loads
#    the parquet/csv this notebook produces. Happy to write it as a
#    follow-up — it shouldn't live inside the scraper notebook.
#
# 3. OA Button: was a useful free aggregator but OA.Works shut it down on
#    2025-11-18. The domain still exists for the website redirect but the
#    API DNS record is gone, hence the NameResolutionErrors. Removed in v3.4.
# ============================================================================

In [ ]:
# ============================================================================
# CELL 7 — OA Full-Text Resolution v3.5.5 MAX  (clean, consolidated, drop-in)
#
# Replaces the v2 Unpaywall-only cell AND every "Cell 7 v3.x" you've pasted
# below it. Delete those — this is the only one you need.
#
# Resolution order (5 providers + 1 title-search fallback):
#   PMC → Unpaywall → Europe PMC → CrossRef → Semantic Scholar → bioRxiv/medRxiv
#   → (preprint title-similarity search if still no full text)
#
# What's new in 3.5 vs 3.4:
#   - NEW provider: SemanticScholarResolver (openAccessPdf field, free, no key)
#     — catches author-hosted PDFs that Unpaywall/CrossRef miss
#   - DEFAULTS flipped to MAX mode:
#       EXTRACT_PDF_TEXT      = True   (downloads + parses PDFs with pypdf)
#       PREPRINT_TITLE_SEARCH = True   (finds preprint twins via Jaccard)
#       PDF_TEXT_MAX_RECORDS  = 100    (was 20)
#       PDF_TEXT_MAX_BYTES    = 12 MB  (was 8 MB)
#   - Coverage report now also shows extracted-content stats, not just links
#
# Expected for a 50-record biomedical query with valid email:
#   - Full-text yield: 65-80% (vs 42% PMC-only)
#   - Extracted text:  ~90% of full-text URLs (some publisher PDFs gate scraping)
#   - Wall time: ~60-90s (PDF downloads dominate; tune PDF_TEXT_MAX_RECORDS down
#     if you want it faster and don't need the actual text)
# ============================================================================
import os, sys, time, logging, re

# ---- 0. Config pulled from earlier cells, with explicit precedence ---------
# Hardcoded so this works even if Cell 2 wasn't re-run / email got reverted.
# If you change your email later, edit this line directly.
USER_EMAIL    = "abrahamtrueba@berkeley.edu"
# (Falls back to globals if you ever blank the line above)
USER_EMAIL    = USER_EMAIL or (globals().get("USER_EMAIL") or globals().get("NCBI_EMAIL") or "").strip()
NCBI_API_KEY  = globals().get("NCBI_API_KEY") or None
ENRICH_UNPAYWALL = bool(globals().get("ENRICH_UNPAYWALL", True))

# v3.5.5 MAX-COVERAGE MODE — these are HARDCODED, ignoring stale globals from
# previous Cell 2 runs. To customize, edit these lines directly.
EXTRACT_PDF_TEXT       = True             # download + parse OA PDFs with pypdf
PDF_TEXT_MAX_BYTES     = 12_000_000       # 12 MB cap per PDF
PDF_TEXT_MAX_RECORDS   = 100              # process up to this many in a batch
PREPRINT_TITLE_SEARCH  = True             # bioRxiv/medRxiv title-similarity fallback
GROQ_SUMMARIES         = bool(globals().get("GROQ_SUMMARIES", False))   # OFF — needs GROQ_API_KEY
GROQ_MODEL             = globals().get("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_MAX_RECORDS       = int(globals().get("GROQ_MAX_RECORDS", 30))

_PLACEHOLDER_EMAILS = {"", "your@email.com", "research@example.com",
                       "you@yourdomain.com", "user@example.com"}
_email_is_real = USER_EMAIL.lower() not in _PLACEHOLDER_EMAILS and "@" in USER_EMAIL

if not _email_is_real:
    print("=" * 72)
    print("  ⚠️  USER_EMAIL IS UNSET OR A PLACEHOLDER")
    print("  ----------------------------------------")
    print("  Unpaywall and CrossRef require a real email per their ToS.")
    print("  Without it, OA yield drops to PMC-only (~40-45% on biomedical sets).")
    print("  With a real email, expect 60-70% on broad PubMed queries.")
    print("")
    print("  Fix: in Cell 2, set")
    print("       NCBI_EMAIL = 'you@your.domain'")
    print("  then re-run this cell. No other changes needed.")
    print("=" * 72)
    print()

# ---- 1. Resolver module (written ONCE) -------------------------------------
_MODULE_PATH = "/content/oa_fulltext_v3.py"
_MODULE_SRC = r'''
from __future__ import annotations
import logging, re, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

log = logging.getLogger("bioscraper.oa_v3")


def _build_session(user_agent="NCBI-BioScraper-Pro/3.5"):
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


def _build_session_no_429_retry(user_agent="NCBI-BioScraper-Pro/3.5"):
    """Like _build_session but does NOT auto-retry 429s. Used for providers
    where we want to see the 429 status code directly (e.g. so a self-throttle
    latch can engage instead of letting urllib3 burn 3 retries silently)."""
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=(500, 502, 503, 504),  # 429 deliberately omitted
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


@dataclass
class OAResult:
    pmid: Optional[str] = None
    doi: Optional[str] = None
    pmcid: Optional[str] = None
    oa_source: str = "none"
    oa_pdf_url: Optional[str] = None
    oa_xml_url: Optional[str] = None
    oa_html_url: Optional[str] = None
    oa_license: Optional[str] = None
    oa_status: str = "closed"
    oa_full_text: bool = False
    oa_attempts: list = field(default_factory=list)
    oa_resolved_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def as_flat_dict(self):
        d = asdict(self)
        d["oa_attempts"] = ",".join(d["oa_attempts"])
        return d


# ---------- providers --------------------------------------------------------
class PMCResolver:
    IDCONV = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    OA_FCGI = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"

    def __init__(self, session, ncbi_api_key=None, tool="ncbi-bioscraper", email="research@example.com"):
        self.s = session; self.api_key = ncbi_api_key; self.tool = tool; self.email = email

    def _params(self, **extra):
        p = {"tool": self.tool, "email": self.email}
        if self.api_key: p["api_key"] = self.api_key
        p.update(extra); return p

    def pmid_to_pmcid(self, pmid, doi=None):
        for key, value in (("pmid", pmid), ("doi", doi)):
            if not value: continue
            try:
                r = self.s.get(self.IDCONV,
                               params=self._params(ids=value, idtype=key, format="json"),
                               timeout=6)
                if r.status_code != 200: continue
                records = r.json().get("records", [])
                if records and records[0].get("pmcid"):
                    return records[0]["pmcid"]
            except Exception as e:
                log.debug("idconv (%s=%s) failed: %s", key, value, e)
        return None

    def pmcid_to_oa(self, pmcid):
        try:
            r = self.s.get(self.OA_FCGI, params={"id": pmcid}, timeout=6)
            if r.status_code != 200 or "<error" in r.text.lower():
                return None
            text = r.text
            pdf_url = xml_url = license_str = None
            for line in text.split("<link "):
                if 'format="pdf"' in line and 'href="' in line:
                    pdf_url = line.split('href="')[1].split('"')[0]
                elif 'format="tgz"' in line and 'href="' in line:
                    xml_url = line.split('href="')[1].split('"')[0]
            if 'license="' in text:
                license_str = text.split('license="')[1].split('"')[0]
            if pdf_url or xml_url:
                if pdf_url and pdf_url.startswith("ftp://"):
                    pdf_url = "https://" + pdf_url[len("ftp://"):]
                if xml_url and xml_url.startswith("ftp://"):
                    xml_url = "https://" + xml_url[len("ftp://"):]
                return {"pdf_url": pdf_url, "xml_url": xml_url,
                        "license": license_str,
                        "html_url": "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % pmcid}
        except Exception as e:
            log.debug("oa.fcgi failed for %s: %s", pmcid, e)
        return None

    def resolve(self, result):
        result.oa_attempts.append("pmc")
        if not result.pmcid:
            result.pmcid = self.pmid_to_pmcid(result.pmid, result.doi)
        if not result.pmcid:
            return False
        oa = self.pmcid_to_oa(result.pmcid)
        if not oa:
            # PMC-hosted but not in OA subset — still useful as a reader URL
            result.oa_html_url = "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % result.pmcid
            return False
        result.oa_source = "pmc"
        result.oa_pdf_url = oa.get("pdf_url")
        result.oa_xml_url = oa.get("xml_url")
        result.oa_html_url = oa.get("html_url")
        result.oa_license = oa.get("license")
        result.oa_status = "gold"
        result.oa_full_text = bool(result.oa_pdf_url or result.oa_xml_url)
        return result.oa_full_text


class UnpaywallResolver:
    BASE = "https://api.unpaywall.org/v2/"

    def __init__(self, session, email):
        self.s = session; self.email = email
        self._not_in_index = 0
        self._auth_failed = False  # latched: stop calling once we see a 422

    def resolve(self, result):
        result.oa_attempts.append("unpaywall")
        if self._auth_failed or not result.doi:
            return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"email": self.email}, timeout=6)
            if r.status_code == 404:
                self._not_in_index += 1
                return False
            if r.status_code == 422:
                # Auth/email problem — latch and stop hammering the API
                self._auth_failed = True
                log.warning("Unpaywall 422 — disabling Unpaywall for the rest of this run "
                            "(check USER_EMAIL is a real address).")
                return False
            if r.status_code != 200:
                return False
            d = r.json()
            if not d.get("is_oa"):
                return False
            best = d.get("best_oa_location") or {}
            if not best:
                locs = d.get("oa_locations") or []
                if locs: best = locs[0]
            if not best: return False
            pdf  = best.get("url_for_pdf")
            html = best.get("url_for_landing_page") or best.get("url")
            if not (pdf or html): return False
            result.oa_source = "unpaywall"
            result.oa_pdf_url = pdf
            result.oa_html_url = html
            result.oa_license = best.get("license")
            result.oa_status = d.get("oa_status") or "bronze"
            result.oa_full_text = bool(pdf)
            return bool(pdf or html)
        except Exception as e:
            log.debug("unpaywall failed for %s: %s", result.doi, e)
            return False


class EuropePMCResolver:
    SEARCH = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def __init__(self, session): self.s = session

    def resolve(self, result):
        result.oa_attempts.append("europepmc")
        if result.pmid:
            query = "EXT_ID:%s AND SRC:MED" % result.pmid
        elif result.doi:
            query = 'DOI:"%s"' % result.doi
        else:
            return False
        try:
            r = self.s.get(self.SEARCH, params={
                "query": query, "format": "json",
                "resultType": "core", "pageSize": 1,
            }, timeout=6)
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            if not hits: return False
            h = hits[0]
            if h.get("isOpenAccess") != "Y": return False
            full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
            pdf = next((u["url"] for u in full_text_urls
                        if u.get("documentStyle", "").lower() == "pdf"
                        and u.get("availability", "").lower() == "open access"), None)
            html = next((u["url"] for u in full_text_urls
                         if u.get("documentStyle", "").lower() == "html"), None)
            if not (pdf or html): return False
            result.oa_source = "europepmc"
            result.oa_pdf_url = pdf or result.oa_pdf_url
            result.oa_html_url = html or result.oa_html_url
            result.oa_license = h.get("license") or result.oa_license
            result.oa_status = "gold" if h.get("inEPMC") == "Y" else "green"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("europepmc failed: %s", e)
            return False


class CrossRefOAResolver:
    """Catches recent papers (<30 days) that Unpaywall hasn't ingested yet,
    and any paper with a CC license declared in CrossRef metadata."""
    BASE = "https://api.crossref.org/works/"

    def __init__(self, session, email):
        self.s = session; self.email = email

    def resolve(self, result):
        result.oa_attempts.append("crossref")
        if not result.doi: return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"mailto": self.email}, timeout=6)
            if r.status_code != 200: return False
            msg = r.json().get("message") or {}
            licenses = msg.get("license") or []
            cc = next((L for L in licenses
                       if "creativecommons.org" in (L.get("URL") or "").lower()), None)
            if not cc: return False
            license_url = cc.get("URL", "")
            license_id = None
            if   "/by-nc-nd" in license_url: license_id = "cc-by-nc-nd"
            elif "/by-nc-sa" in license_url: license_id = "cc-by-nc-sa"
            elif "/by-nc"    in license_url: license_id = "cc-by-nc"
            elif "/by-sa"    in license_url: license_id = "cc-by-sa"
            elif "/by"       in license_url: license_id = "cc-by"
            elif "/zero"     in license_url: license_id = "cc0"
            pdf = None
            for link in (msg.get("link") or []):
                if (link.get("content-type") == "application/pdf"
                        and "vor" in (link.get("intended-application", "") or "").lower()):
                    pdf = link.get("URL"); break
            result.oa_source = "crossref"
            result.oa_pdf_url = pdf
            result.oa_html_url = "https://doi.org/" + result.doi
            result.oa_license = license_id or "cc-detected"
            result.oa_status = "gold"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("crossref failed for %s: %s", result.doi, e)
            return False


class SemanticScholarResolver:
    """Semantic Scholar's openAccessPdf field. Free, no key needed at low
    volume (~1 req/sec unauthenticated). Catches a different slice than
    Unpaywall/CrossRef — especially newer papers and CS/biology preprints
    where their crawler picks up author-hosted PDFs.

    Self-rate-limits to 1 req/sec to avoid 429 cascades.
    Skips itself entirely if `result.oa_full_text` is already True from an
    earlier provider — saves quota for records that actually need it.
    Uses its own session that does NOT retry 429s, so the latch can engage
    on the first 429 instead of letting urllib3 burn 3 silent retries.
    """
    BASE = "https://api.semanticscholar.org/graph/v1/paper/"
    MIN_INTERVAL = 1.05  # seconds between calls

    def __init__(self, session=None, email=None):
        # Ignore the shared session — S2 needs the no-retry-on-429 behaviour
        self.s = _build_session_no_429_retry()
        self.email = email
        self._last_call = 0.0
        self._rate_limited = False  # latch after a 429 to stop hammering

    def _lookup(self, identifier):
        if self._rate_limited:
            return None
        # Self-throttle to 1 req/sec
        elapsed = time.time() - self._last_call
        if elapsed < self.MIN_INTERVAL:
            time.sleep(self.MIN_INTERVAL - elapsed)
        self._last_call = time.time()
        try:
            r = self.s.get(self.BASE + identifier,
                           params={"fields": "openAccessPdf,externalIds,isOpenAccess,title"},
                           timeout=6)
            if r.status_code == 429:
                # Latch off — S2 has decided we've had enough. Don't keep retrying.
                self._rate_limited = True
                log.warning("Semantic Scholar 429 — disabling S2 for the rest of this run.")
                return None
            if r.status_code != 200:
                return None
            return r.json()
        except Exception as e:
            # If urllib3 still managed to hit retry exhaustion on 429s
            # (shouldn't with our no-retry session, but belt-and-braces),
            # treat that as a hard latch too.
            err_str = str(e).lower()
            if "429" in err_str or "too many" in err_str:
                self._rate_limited = True
                log.warning("Semantic Scholar repeated 429 — disabling S2 for the rest of this run.")
            else:
                log.debug("semanticscholar lookup %s failed: %s", identifier, e)
            return None

    def resolve(self, result):
        # Skip entirely if an earlier provider already gave us a PDF
        if result.oa_full_text:
            return False
        result.oa_attempts.append("semanticscholar")
        d = None
        if result.doi:
            d = self._lookup("DOI:" + result.doi)
        if not d and result.pmid:
            d = self._lookup("PMID:" + result.pmid)
        if not d:
            return False
        oa_pdf = d.get("openAccessPdf") or {}
        url = oa_pdf.get("url")
        if not url:
            return False
        result.oa_source = "semanticscholar"
        result.oa_pdf_url = url
        result.oa_html_url = result.oa_html_url or url
        result.oa_license = oa_pdf.get("license") or result.oa_license or "s2-detected"
        if result.oa_status == "closed":
            result.oa_status = "green"
        result.oa_full_text = True
        return True


# OAButtonResolver removed in v3.4: OA.Works retired the openaccessbutton.org
# API on 2025-11-18. Calls were 100% failing with DNS errors. If you want a
# similar drop-in replacement, CORE (api.core.ac.uk, free key) or Semantic
# Scholar's openAccessPdf field both work and follow the same `resolve(result)`
# contract.


class PreprintResolver:
    """Direct lookup for 10.1101/* DOIs (bioRxiv/medRxiv)."""
    def __init__(self, session): self.s = session


    def resolve(self, result):
        result.oa_attempts.append("preprint")
        if not result.doi or not result.doi.startswith("10.1101/"):
            return False
        for host in ("www.biorxiv.org", "www.medrxiv.org"):
            url = "https://%s/content/%s.full.pdf" % (host, result.doi)
            try:
                r = self.s.head(url, timeout=8, allow_redirects=True)
                if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
                    result.oa_source = "preprint"
                    result.oa_pdf_url = url
                    result.oa_html_url = url.rsplit(".full.pdf", 1)[0] + ".full"
                    result.oa_status = "green"
                    result.oa_full_text = True
                    return True
            except Exception:
                continue
        return False


class PreprintTitleSearchResolver:
    """Last-resort: search bioRxiv/medRxiv by title and accept matches with
    high token-overlap similarity. Useful when a peer-reviewed paper has
    a preprint twin under a different DOI."""
    BIORXIV_API = "https://api.biorxiv.org/details/{server}/{date_or_doi}/0/json"
    SEARCH = "https://www.biorxiv.org/search/{query}"  # HTML, not used directly

    def __init__(self, session, min_similarity=0.75):
        self.s = session
        self.min_similarity = min_similarity

    @staticmethod
    def _norm_tokens(s):
        s = (s or "").lower()
        s = re.sub(r"[^a-z0-9 ]+", " ", s)
        return set(t for t in s.split() if len(t) > 2)

    def _similarity(self, a, b):
        ta, tb = self._norm_tokens(a), self._norm_tokens(b)
        if not ta or not tb: return 0.0
        return len(ta & tb) / len(ta | tb)  # Jaccard

    def resolve(self, result, candidate_title=None):
        result.oa_attempts.append("preprint_search")
        if not candidate_title: return False
        # bioRxiv has no native title-search API; we use Europe PMC restricted
        # to preprint sources, which DOES index bioRxiv/medRxiv records.
        try:
            r = self.s.get(
                "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                params={"query": 'TITLE:"%s" AND (SRC:PPR)' % candidate_title[:200],
                        "format": "json", "resultType": "core", "pageSize": 5},
                timeout=6,
            )
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            for h in hits:
                sim = self._similarity(candidate_title, h.get("title"))
                if sim < self.min_similarity: continue
                doi = h.get("doi")
                full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
                pdf = next((u["url"] for u in full_text_urls
                            if u.get("documentStyle", "").lower() == "pdf"), None)
                html = next((u["url"] for u in full_text_urls
                             if u.get("documentStyle", "").lower() == "html"), None)
                if not (pdf or html): continue
                result.oa_source = "preprint_search"
                result.oa_pdf_url = pdf or result.oa_pdf_url
                result.oa_html_url = html or result.oa_html_url
                result.oa_status = "green"
                result.oa_full_text = bool(pdf)
                return True
        except Exception as e:
            log.debug("preprint_search failed: %s", e)
        return False


# ---------- orchestrator -----------------------------------------------------
class OAFullTextResolver:
    def __init__(self, email, ncbi_api_key=None,
                 enable_pmc=True, enable_unpaywall=True,
                 enable_europepmc=True, enable_crossref=True,
                 enable_semanticscholar=True,
                 enable_oabutton=False,  # kept for back-compat; ignored (API retired)
                 enable_preprint=True,
                 enable_preprint_search=False,
                 polite_delay_s=0.0):
        self.session = _build_session()
        self.polite_delay = polite_delay_s
        self.providers = []
        if enable_pmc:
            self.providers.append(PMCResolver(self.session, ncbi_api_key, email=email))
        if enable_unpaywall:
            self.providers.append(UnpaywallResolver(self.session, email=email))
        if enable_europepmc:
            self.providers.append(EuropePMCResolver(self.session))
        if enable_crossref:
            self.providers.append(CrossRefOAResolver(self.session, email=email))
        if enable_semanticscholar:
            self.providers.append(SemanticScholarResolver(self.session, email=email))
        # OA Button removed in v3.4 (api.openaccessbutton.org retired 2025-11-18)
        if enable_preprint:
            self.providers.append(PreprintResolver(self.session))
        # Title-search resolver is held separately because it needs a title arg
        self.preprint_search = (PreprintTitleSearchResolver(self.session)
                                if enable_preprint_search else None)

    def resolve_one(self, pmid=None, doi=None, pmcid=None, title=None):
        result = OAResult(pmid=pmid, doi=doi, pmcid=pmcid)
        for provider in self.providers:
            try:
                if provider.resolve(result) and result.oa_full_text:
                    break
            except Exception as e:
                log.warning("%s raised: %s", type(provider).__name__, e)
            if self.polite_delay:
                time.sleep(self.polite_delay)
        # Title-similarity fallback only if we still don't have full text
        if (self.preprint_search is not None
                and not result.oa_full_text
                and title):
            try:
                self.preprint_search.resolve(result, candidate_title=title)
            except Exception as e:
                log.warning("PreprintTitleSearchResolver raised: %s", e)
        return result

    def enrich_dataframe(self, df, pmid_col="pmid", doi_col="doi",
                         pmcid_col=None, title_col=None, show_progress=True):
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(df.iterrows(), total=len(df),
                            desc="OA resolve", disable=not show_progress)
        except ImportError:
            iterator = df.iterrows()

        rows = []
        import pandas as pd
        for _, row in iterator:
            def _get(col):
                if col and col in df.columns:
                    v = row[col]
                    if v is None: return None
                    try:
                        if pd.isna(v): return None
                    except Exception: pass
                    s = str(v).strip()
                    return s if s and s.lower() != "nan" else None
                return None
            rows.append(self.resolve_one(
                pmid=_get(pmid_col),
                doi=_get(doi_col),
                pmcid=_get(pmcid_col),
                title=_get(title_col),
            ).as_flat_dict())

        oa_df = pd.DataFrame(rows)
        # Don't re-emit identifier cols already present on the input df, otherwise
        # a second pass through enrich_dataframe will collide on pmid/doi/pmcid.
        for c in ("pmid", "doi", "pmcid"):
            if c in oa_df.columns and c in df.columns:
                oa_df = oa_df.drop(columns=[c])
        for c in list(df.columns):
            if c.startswith("oa_") or c.startswith("up_"):
                df = df.drop(columns=[c])
        return df.reset_index(drop=True).join(oa_df.reset_index(drop=True))

    def coverage_report(self, df):
        total = len(df)
        if total == 0: return {"total": 0}
        return {
            "total": total,
            "full_text_pct": round(100 * df["oa_full_text"].sum() / total, 1),
            "any_oa_link_pct": round(100 * (df["oa_source"] != "none").sum() / total, 1),
            "by_source": df["oa_source"].value_counts().to_dict(),
            "by_status": df["oa_status"].value_counts().to_dict(),
        }
'''

# Write it ONCE, force a clean import
with open(_MODULE_PATH, "w", encoding="utf-8") as f:
    f.write(_MODULE_SRC)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
for _mod in [m for m in sys.modules if m.startswith("oa_fulltext_v3")]:
    del sys.modules[_mod]

from oa_fulltext_v3 import OAFullTextResolver, UnpaywallResolver  # noqa: E402

# ---- 2. Build resolver with sane provider gating ---------------------------
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("bioscraper.oa_v3").setLevel(logging.WARNING)  # quiet the per-record DEBUG noise

oa = OAFullTextResolver(
    email=USER_EMAIL or "research@example.com",
    ncbi_api_key=NCBI_API_KEY,
    enable_pmc=True,
    enable_unpaywall=ENRICH_UNPAYWALL and _email_is_real,   # << the fix
    enable_europepmc=True,
    enable_crossref=_email_is_real,                          # CrossRef wants mailto too
    # OA Button retired 2025-11-18, kwarg dropped
    enable_preprint=True,
    enable_preprint_search=PREPRINT_TITLE_SEARCH,
)

# ---- 3. Enrich -------------------------------------------------------------
# Self-heal: if `df` was knocked out of the namespace by a prior error or
# kernel reset, rebuild it from the SQLite cache from earlier scrapes.
def _rebuild_df_from_cache():
    """Rebuild a minimal df (pmid/doi/title) from the pubmed_cache SQLite table.
    Only enough columns for Cell 7 to operate. Does not re-fetch from PubMed."""
    import sqlite3, json as _json
    import pandas as _pd

    candidate_paths = []
    if "CACHE_DB" in globals():
        candidate_paths.append(globals()["CACHE_DB"])
    if "OUTPUT_DIR" in globals():
        candidate_paths.append(os.path.join(globals()["OUTPUT_DIR"], "_cache.sqlite"))
    # Common defaults if config globals also got wiped
    candidate_paths += [
        "/content/output/_cache.sqlite",
        "/content/drive/MyDrive/NCBI_Research/crispr_cancer_v2/_cache.sqlite",
    ]

    db_path = next((p for p in candidate_paths if p and os.path.exists(p)), None)
    if not db_path:
        return None, "no cache file found at: " + ", ".join(str(p) for p in candidate_paths)

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.execute("SELECT pmid, record_json FROM pubmed_cache")
        rows = []
        for pmid, blob in cursor.fetchall():
            try:
                rec = _json.loads(blob) if blob else {}
            except Exception:
                rec = {}
            doi = (rec.get("LID") or rec.get("AID") or "")
            if isinstance(doi, list): doi = doi[0] if doi else ""
            doi = str(doi).strip()
            # Strip trailing "[doi]" tag PubMed sometimes attaches
            doi = re.sub(r"\s*\[doi\]\s*$", "", doi, flags=re.IGNORECASE).strip()
            # Keep only DOIs that look like DOIs (start with 10.)
            if doi and not doi.startswith("10."):
                m = re.search(r"\b(10\.\d{4,9}/[^\s]+)", doi)
                doi = m.group(1) if m else ""
            rows.append({
                "pmid":  str(pmid),
                "doi":   doi,
                "title": str(rec.get("TI", "")),
                "abstract": str(rec.get("AB", "")),
                "pmcid": str(rec.get("PMC", "") or ""),
            })
        conn.close()
        if not rows:
            return None, f"cache at {db_path} has 0 records"
        return _pd.DataFrame(rows), f"loaded {len(rows)} records from {db_path}"
    except Exception as e:
        return None, f"cache read failed at {db_path}: {e}"


def _fresh_pubmed_scrape(query, max_results=50, start_year=None, end_year=None,
                         email=None, api_key=None):
    """Minimal self-contained PubMed scrape using NCBI E-utilities REST API.
    Only pulls the fields needed for OA resolution: pmid, doi, title, abstract,
    pmcid. No Biopython dependency. Returns a DataFrame or None on failure.
    """
    import urllib.parse, requests as _rq
    import pandas as _pd
    from xml.etree import ElementTree as ET

    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    session = _rq.Session()
    session.headers.update({"User-Agent": "NCBI-BioScraper-Pro/3.5.5"})

    # Build query with date filter if given
    full_query = query
    if start_year and end_year:
        full_query += f" AND ({start_year}[PDAT]:{end_year}[PDAT])"

    # 1. esearch — get PMIDs
    esearch_params = {"db": "pubmed", "term": full_query, "retmax": max_results,
                      "retmode": "json"}
    if email:   esearch_params["email"] = email
    if api_key: esearch_params["api_key"] = api_key

    try:
        r = session.get(base + "esearch.fcgi", params=esearch_params, timeout=15)
        r.raise_for_status()
        ids = r.json().get("esearchresult", {}).get("idlist", []) or []
    except _rq.exceptions.HTTPError as e:
        status = e.response.status_code if e.response is not None else "?"
        print(f"   ❌ esearch failed (HTTP {status}): {e}")
        if status == 429:
            print("       NCBI is rate-limiting you. Wait 60s and retry.")
        elif status == 403:
            print("       NCBI is blocking the source IP. Set NCBI_API_KEY in Cell 2")
            print("       to authenticate, or wait if you've been hitting rate limits.")
        return None
    except Exception as e:
        print(f"   ❌ esearch failed: {type(e).__name__}: {e}")
        return None
    if not ids:
        print(f"   ❌ esearch returned 0 PMIDs for query: {full_query!r}")
        return None
    print(f"   esearch: {len(ids)} PMIDs")

    # 2. efetch — get full records (XML for structured DOI/PMC extraction)
    efetch_params = {"db": "pubmed", "id": ",".join(ids),
                     "rettype": "xml", "retmode": "xml"}
    if email:   efetch_params["email"] = email
    if api_key: efetch_params["api_key"] = api_key

    try:
        r = session.get(base + "efetch.fcgi", params=efetch_params, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
    except Exception as e:
        print(f"   ❌ efetch failed: {e}")
        return None

    rows = []
    for article in root.iter("PubmedArticle"):
        pmid_elem = article.find(".//PMID")
        pmid = pmid_elem.text if pmid_elem is not None else ""

        title_elem = article.find(".//ArticleTitle")
        title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

        # Abstract may be split across multiple AbstractText elements
        abstract_parts = []
        for at in article.iter("AbstractText"):
            t = "".join(at.itertext()).strip()
            if t: abstract_parts.append(t)
        abstract = " ".join(abstract_parts)

        # DOI lives in ArticleIdList as <ArticleId IdType="doi">
        doi = ""; pmcid = ""
        for aid in article.iter("ArticleId"):
            id_type = (aid.get("IdType") or "").lower()
            val = (aid.text or "").strip()
            if id_type == "doi" and not doi: doi = val
            elif id_type == "pmc" and not pmcid: pmcid = val

        rows.append({
            "pmid":     str(pmid),
            "doi":      doi,
            "title":    title,
            "abstract": abstract,
            "pmcid":    pmcid,
        })

    if not rows:
        print(f"   ❌ efetch returned 0 parsable articles")
        return None
    return _pd.DataFrame(rows)


if "df" not in globals() or globals().get("df") is None:
    print("⚠️  `df` not in scope — attempting recovery…\n")
    import pandas as pd

    # Attempt 1: rebuild from SQLite cache (free, instant)
    print("   [1/2] Trying SQLite cache…")
    rebuilt, msg = _rebuild_df_from_cache()
    print(f"         {msg}")

    if rebuilt is not None and len(rebuilt) > 0:
        df = rebuilt
        print(f"   ✅ Rebuilt from cache: {len(df)} records")
    else:
        # Attempt 2: fresh PubMed scrape using config from Cell 2
        query        = globals().get("SEARCH_QUERY",  "CRISPR cancer therapy")
        max_results  = globals().get("MAX_RESULTS",   50)
        start_year   = globals().get("START_YEAR",    None)
        end_year     = globals().get("END_YEAR",      None)
        print(f"\n   [2/2] Cache empty — running fresh PubMed scrape:")
        print(f"         query='{query}' | max={max_results} | years={start_year}-{end_year}")
        rebuilt = _fresh_pubmed_scrape(query=query, max_results=max_results,
                                       start_year=start_year, end_year=end_year,
                                       email=USER_EMAIL if _email_is_real else None,
                                       api_key=NCBI_API_KEY)
        if rebuilt is None or len(rebuilt) == 0:
            print("\n❌ Could not recover `df` from either cache or fresh scrape.")
            print("   Check your network and SEARCH_QUERY config in Cell 2.")
            raise RuntimeError("df missing and recovery failed")
        df = rebuilt
        print(f"   ✅ Fresh scrape complete: {len(df)} records")

    print(f"   Columns: {list(df.columns)}")
    print(f"   With DOI:  {(df['doi'] != '').sum()}/{len(df)}")
    print(f"   With PMCID:{(df['pmcid'] != '').sum()}/{len(df)}")
    print()

# Use 'title' column if it exists, for the preprint title-similarity fallback
_title_col = "title" if "title" in df.columns else None
df = df.drop(columns=[c for c in df.columns if c.startswith("oa_") or c.startswith("up_")],
             errors="ignore")
df = oa.enrich_dataframe(df, pmid_col="pmid", doi_col="doi", title_col=_title_col)

# ---- 4. Coverage report ----------------------------------------------------
report = oa.coverage_report(df)
print("\n📊 OA full-text resolution v3.5.5 MAX (%d records)" % report["total"])
print("   Full-text yield: %s%%" % report["full_text_pct"])
print("   Any OA link:     %s%%" % report["any_oa_link_pct"])
print("   By source:       %s" % report["by_source"])
print("   By status:       %s" % report["by_status"])

unpaywall_obj = next((p for p in oa.providers if isinstance(p, UnpaywallResolver)), None)
if unpaywall_obj is not None:
    n_miss = unpaywall_obj._not_in_index
    print(f"\n   Unpaywall index misses (404s): {n_miss}/{len(df)}")
    print( "   -> these are typically <30-day-old DOIs not yet ingested by Unpaywall")
    if unpaywall_obj._auth_failed:
        print("   ⚠️  Unpaywall auth failed mid-run — fix USER_EMAIL and re-run this cell.")
elif ENRICH_UNPAYWALL and not _email_is_real:
    print("\n   Unpaywall: SKIPPED (placeholder email).")


# ============================================================================
# CELL 7b — Download PDFs and extract plain text
# Controlled by EXTRACT_PDF_TEXT flag. ON by default in v3.5 MAX.
# Uses pypdf only (no GPU, no OCR). Caps at PDF_TEXT_MAX_RECORDS for safety.
# ============================================================================
print()
print("─" * 72)
print(f"  PDF text extraction: {'ON' if EXTRACT_PDF_TEXT else 'OFF'}  "
      f"(EXTRACT_PDF_TEXT = {EXTRACT_PDF_TEXT})")
if EXTRACT_PDF_TEXT:
    print(f"  Cap: {PDF_TEXT_MAX_RECORDS} PDFs, "
          f"{PDF_TEXT_MAX_BYTES // 1_000_000} MB each")
print("─" * 72)

if EXTRACT_PDF_TEXT:
    try:
        import pypdf  # noqa: F401
    except ImportError:
        print("Installing pypdf for text extraction...")
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
        import pypdf

    import io, requests as _rq
    from collections import Counter as _Counter
    from pypdf import PdfReader

    # Browser-style User-Agent — many publishers block obvious scrapers but
    # serve OA content fine to a normal-looking browser fetch. This is OA
    # content the publisher has affirmatively licensed for free reading.
    _PDF_UA = ("Mozilla/5.0 (X11; Linux x86_64) "
               "AppleWebKit/537.36 (KHTML, like Gecko) "
               "Chrome/124.0.0.0 Safari/537.36")
    _PDF_HEADERS = {
        "User-Agent": _PDF_UA,
        "Accept": "application/pdf,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    # Track precise failure reasons so we can SEE why extraction is failing
    # rather than guessing. This is the key change in v3.5.3.
    _failure_reasons = []   # list of (idx, source, url, reason, detail)

    def _extract_pdf_text(url, max_bytes=PDF_TEXT_MAX_BYTES, idx=None, source=None):
        if not url:
            _failure_reasons.append((idx, source, url, "no_url", ""))
            return None
        try:
            with _rq.get(url, stream=True, timeout=20,
                         headers=_PDF_HEADERS, allow_redirects=True) as r:
                if r.status_code != 200:
                    _failure_reasons.append(
                        (idx, source, url, f"http_{r.status_code}",
                         f"final_url={r.url}"))
                    return None
                ctype = (r.headers.get("content-type") or "").lower()
                final_url = r.url
                if "pdf" not in ctype and not final_url.lower().endswith(".pdf"):
                    # Likely redirected to a login wall or HTML reader page
                    _failure_reasons.append(
                        (idx, source, url, "not_pdf", f"ctype={ctype} final={final_url}"))
                    return None
                buf = io.BytesIO()
                total = 0
                for chunk in r.iter_content(chunk_size=65536):
                    if not chunk: continue
                    buf.write(chunk); total += len(chunk)
                    if total > max_bytes:
                        _failure_reasons.append(
                            (idx, source, url, "too_big", f"{total:,} bytes"))
                        return None
                buf.seek(0)
                try:
                    reader = PdfReader(buf)
                except Exception as e:
                    _failure_reasons.append(
                        (idx, source, url, "pdf_parse_error", str(e)[:120]))
                    return None
                pages = []
                for page in reader.pages:
                    try: pages.append(page.extract_text() or "")
                    except Exception: pass
                text = "\n\n".join(pages).strip()
                text = re.sub(r"[ \t]+", " ", text)
                text = re.sub(r"\n{3,}", "\n\n", text)
                if not text:
                    _failure_reasons.append(
                        (idx, source, url, "empty_text", f"{total:,} bytes downloaded"))
                    return None
                return text
        except _rq.exceptions.Timeout:
            _failure_reasons.append((idx, source, url, "timeout", ""))
        except _rq.exceptions.SSLError as e:
            _failure_reasons.append((idx, source, url, "ssl_error", str(e)[:120]))
        except Exception as e:
            _failure_reasons.append((idx, source, url, "exception", str(e)[:120]))
        return None

    eligible = df[df["oa_full_text"] & df["oa_pdf_url"].notna()].copy()
    priority_order = ["pmc", "preprint", "preprint_search", "europepmc",
                      "unpaywall", "semanticscholar", "crossref"]
    eligible["_prio"] = eligible["oa_source"].map(
        {s: i for i, s in enumerate(priority_order)}).fillna(99)
    eligible = eligible.sort_values("_prio").head(PDF_TEXT_MAX_RECORDS)

    # Diagnostic context: report what we're processing and what we're skipping
    n_full_text  = int(df["oa_full_text"].sum())
    n_with_pdf   = int(df["oa_pdf_url"].notna().sum())
    n_xml_only   = int((df["oa_full_text"] & df["oa_pdf_url"].isna() &
                        df["oa_xml_url"].notna()).sum())

    if len(eligible) == 0:
        print("\n  ⚠️  No records with full-text PDF URLs to extract.")
        print("     (Need oa_full_text=True AND oa_pdf_url not null.)")
        df["oa_full_text_extracted"] = None
        df["oa_full_text_chars"] = 0
    else:
        print(f"\n📄 Extraction queue ({len(eligible)} of {n_full_text} full-text records):")
        print(f"   Records flagged full_text=True:    {n_full_text}")
        print(f"   Records with a PDF URL:            {n_with_pdf}")
        if n_xml_only:
            print(f"   Records with XML/tgz only (skipped): {n_xml_only}")
        print(f"   By source in queue: {dict(eligible['oa_source'].value_counts())}")

        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(eligible.iterrows(), total=len(eligible), desc="PDF extract")
        except ImportError:
            it = eligible.iterrows()

        texts = {}
        for idx, row in it:
            t = _extract_pdf_text(row["oa_pdf_url"], idx=idx, source=row["oa_source"])
            if t: texts[idx] = t

        df["oa_full_text_extracted"] = df.index.map(texts).where(df.index.isin(texts), None)
        df["oa_full_text_chars"]     = df["oa_full_text_extracted"].fillna("").str.len()
        n_extracted = int(df["oa_full_text_chars"].gt(0).sum())
        median_chars = int(df.loc[df["oa_full_text_chars"] > 0, "oa_full_text_chars"].median() or 0)
        print(f"\n   Extracted: {n_extracted}/{len(eligible)} PDFs successfully parsed")
        if n_extracted:
            print(f"   Median length: {median_chars:,} chars (~{median_chars//5_000} pages of plain text)")
            print(f"   Total content captured: {int(df['oa_full_text_chars'].sum()):,} chars")

        # Per-source success rate — tells us which providers gave usable PDFs
        succ_by_src = df.loc[df["oa_full_text_chars"] > 0, "oa_source"].value_counts().to_dict()
        attempted_by_src = eligible["oa_source"].value_counts().to_dict()
        print(f"\n   Per-source extraction success:")
        for src in attempted_by_src:
            ok = succ_by_src.get(src, 0); tot = attempted_by_src[src]
            print(f"     {src:18s} {ok}/{tot}")

        # Failure-reason breakdown — the diagnostic that was missing
        if _failure_reasons:
            reason_counts = _Counter(fr[3] for fr in _failure_reasons)
            print(f"\n   Failure reasons ({len(_failure_reasons)} total):")
            for reason, count in reason_counts.most_common():
                print(f"     {reason:18s} {count}")
            # Show first few examples per reason category
            print(f"\n   Failure examples (up to 5):")
            for fr in _failure_reasons[:5]:
                idx_, src_, url_, reason_, detail_ = fr
                short_url = url_[:60] + "…" if url_ and len(url_) > 60 else (url_ or "")
                print(f"     [{src_}] {reason_}: {short_url}")
                if detail_: print(f"       └─ {detail_[:140]}")

        # Stash the failure log on df for post-hoc inspection
        df.attrs["pdf_extract_failures"] = _failure_reasons

        print(f"\n   Records with ABSTRACT only:        {len(df) - n_extracted}")
        print(f"   Records with FULL TEXT extracted:  {n_extracted}")
        print(f"   -> use df['oa_full_text_extracted'] for downstream NLP / LLM input")
        print(f"   -> use df.attrs['pdf_extract_failures'] to inspect failed downloads")


# ============================================================================
# CELL 7c — Optional: Groq LLM summaries + key findings
# Off by default. Requires:  pip install groq   AND   os.environ["GROQ_API_KEY"]
# Operates on EXTRACTED full text if available, else on the abstract.
# ============================================================================
if GROQ_SUMMARIES:
    api_key = os.environ.get("GROQ_API_KEY", "").strip()
    if not api_key:
        print("\n⚠️  GROQ_SUMMARIES=True but GROQ_API_KEY not set in environment. Skipping.")
    else:
        try:
            from groq import Groq  # noqa: F401
        except ImportError:
            print("Installing groq client…")
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
        from groq import Groq

        client = Groq(api_key=api_key)

        SYSTEM_PROMPT = (
            "You are summarising a biomedical paper for a domain expert in "
            "molecular biology, genomics, and computational biology. "
            "Return STRICT JSON with exactly these keys: "
            '{"tldr": str (<=40 words), '
            '"key_findings": list[str] (3-5 bullets, each <=25 words), '
            '"methods": str (<=30 words), '
            '"caveats": str (<=25 words, or empty string if none)}. '
            "Stay concrete. Cite figures or numbers when they appear in the source. "
            "Do not invent details. Output ONLY the JSON object."
        )

        def _summarise_one(text, model=GROQ_MODEL, max_chars=24000):
            text = (text or "").strip()
            if not text: return None
            payload = text[:max_chars]
            try:
                resp = client.chat.completions.create(
                    model=model,
                    temperature=0.2,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": payload},
                    ],
                )
                return resp.choices[0].message.content
            except Exception as e:
                logging.getLogger("bioscraper.oa_v3").warning("Groq summarise failed: %s", e)
                return None

        # Choose source text: extracted full text > abstract
        has_fulltext = ("oa_full_text_extracted" in df.columns)
        candidates = df.copy()
        if has_fulltext:
            candidates["_src"] = candidates["oa_full_text_extracted"].fillna(
                candidates.get("abstract", ""))
        else:
            candidates["_src"] = candidates.get("abstract", "")
        candidates = candidates[candidates["_src"].fillna("").str.len() > 200]
        candidates = candidates.head(GROQ_MAX_RECORDS)

        print(f"\n🤖 Summarising {len(candidates)} records via Groq ({GROQ_MODEL})…")
        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(candidates.iterrows(), total=len(candidates), desc="LLM summarise")
        except ImportError:
            it = candidates.iterrows()

        summaries = {}
        for idx, row in it:
            s = _summarise_one(row["_src"])
            if s: summaries[idx] = s

        df["llm_summary_json"] = df.index.map(summaries).where(df.index.isin(summaries), None)
        # Lightly parse for convenience
        import json as _json
        def _safe_json(s):
            try: return _json.loads(s) if s else None
            except Exception: return None
        parsed = df["llm_summary_json"].map(_safe_json)
        df["llm_tldr"]         = parsed.map(lambda d: (d or {}).get("tldr"))
        df["llm_key_findings"] = parsed.map(lambda d: " | ".join((d or {}).get("key_findings") or []))
        df["llm_methods"]      = parsed.map(lambda d: (d or {}).get("methods"))
        df["llm_caveats"]      = parsed.map(lambda d: (d or {}).get("caveats"))

        n_ok = df["llm_tldr"].notna().sum()
        print(f"   Summaries produced: {n_ok}/{len(candidates)}")


# ============================================================================
# Notes that are NOT in the code, on purpose:
#
# 1. Sci-Hub fallback was suggested in the feedback. I deliberately did not
#    add it. Sci-Hub redistributes copyrighted papers without authorisation;
#    bundling it into a notebook you plan to publish would expose you (and
#    anyone forking the repo) to DMCA/takedown risk and would get the repo
#    flagged on GitHub. The legitimate stack above (PMC + Unpaywall +
#    Europe PMC + CrossRef + bioRxiv) is what actually gets you to the
#    60–70% yield range your reviewer mentioned, AS LONG AS USER_EMAIL is
#    set. If you want one more legitimate boost, add CORE (api.core.ac.uk,
#    free key) or Semantic Scholar's openAccessPdf field — both are
#    drop-in providers using the same `resolve(result)` contract.
#
# 2. Streamlit dashboard: that's a separate deliverable, not a Cell 7 fix.
#    The right move is a `streamlit_app.py` in the repo root that loads
#    the parquet/csv this notebook produces. Happy to write it as a
#    follow-up — it shouldn't live inside the scraper notebook.
#
# 3. OA Button: was a useful free aggregator but OA.Works shut it down on
#    2025-11-18. The domain still exists for the website redirect but the
#    API DNS record is gone, hence the NameResolutionErrors. Removed in v3.4.
# ============================================================================

In [ ]:
# ============================================================================
# CELL 7 — OA Full-Text Resolution v3.5.7 MAX  (clean, consolidated, drop-in)
#
# Replaces the v2 Unpaywall-only cell AND every "Cell 7 v3.x" you've pasted
# below it. Delete those — this is the only one you need.
#
# Resolution order (5 providers + 1 title-search fallback):
#   PMC → Unpaywall → Europe PMC → CrossRef → Semantic Scholar → bioRxiv/medRxiv
#   → (preprint title-similarity search if still no full text)
#
# What's new in 3.5 vs 3.4:
#   - NEW provider: SemanticScholarResolver (openAccessPdf field, free, no key)
#     — catches author-hosted PDFs that Unpaywall/CrossRef miss
#   - DEFAULTS flipped to MAX mode:
#       EXTRACT_PDF_TEXT      = True   (downloads + parses PDFs with pypdf)
#       PREPRINT_TITLE_SEARCH = True   (finds preprint twins via Jaccard)
#       PDF_TEXT_MAX_RECORDS  = 100    (was 20)
#       PDF_TEXT_MAX_BYTES    = 12 MB  (was 8 MB)
#   - Coverage report now also shows extracted-content stats, not just links
#
# Expected for a 50-record biomedical query with valid email:
#   - Full-text yield: 65-80% (vs 42% PMC-only)
#   - Extracted text:  ~90% of full-text URLs (some publisher PDFs gate scraping)
#   - Wall time: ~60-90s (PDF downloads dominate; tune PDF_TEXT_MAX_RECORDS down
#     if you want it faster and don't need the actual text)
# ============================================================================
import os, sys, time, logging, re

# ---- 0. Config pulled from earlier cells, with explicit precedence ---------
# Hardcoded so this works even if Cell 2 wasn't re-run / email got reverted.
# If you change your email later, edit this line directly.
USER_EMAIL    = "abrahamtrueba@berkeley.edu"
# (Falls back to globals if you ever blank the line above)
USER_EMAIL    = USER_EMAIL or (globals().get("USER_EMAIL") or globals().get("NCBI_EMAIL") or "").strip()
NCBI_API_KEY  = globals().get("NCBI_API_KEY") or None
ENRICH_UNPAYWALL = bool(globals().get("ENRICH_UNPAYWALL", True))

# v3.5.7 MAX-COVERAGE MODE — these are HARDCODED, ignoring stale globals from
# previous Cell 2 runs. To customize, edit these lines directly.
EXTRACT_PDF_TEXT       = True             # download + parse OA PDFs with pypdf
PDF_TEXT_MAX_BYTES     = 12_000_000       # 12 MB cap per PDF
PDF_TEXT_MAX_RECORDS   = 100              # process up to this many in a batch
PREPRINT_TITLE_SEARCH  = True             # bioRxiv/medRxiv title-similarity fallback
GROQ_SUMMARIES         = bool(globals().get("GROQ_SUMMARIES", False))   # OFF — needs GROQ_API_KEY
GROQ_MODEL             = globals().get("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_MAX_RECORDS       = int(globals().get("GROQ_MAX_RECORDS", 30))

_PLACEHOLDER_EMAILS = {"", "your@email.com", "research@example.com",
                       "you@yourdomain.com", "user@example.com"}
_email_is_real = USER_EMAIL.lower() not in _PLACEHOLDER_EMAILS and "@" in USER_EMAIL

if not _email_is_real:
    print("=" * 72)
    print("  ⚠️  USER_EMAIL IS UNSET OR A PLACEHOLDER")
    print("  ----------------------------------------")
    print("  Unpaywall and CrossRef require a real email per their ToS.")
    print("  Without it, OA yield drops to PMC-only (~40-45% on biomedical sets).")
    print("  With a real email, expect 60-70% on broad PubMed queries.")
    print("")
    print("  Fix: in Cell 2, set")
    print("       NCBI_EMAIL = 'you@your.domain'")
    print("  then re-run this cell. No other changes needed.")
    print("=" * 72)
    print()

# ---- 1. Resolver module (written ONCE) -------------------------------------
_MODULE_PATH = "/content/oa_fulltext_v3.py"
_MODULE_SRC = r'''
from __future__ import annotations
import logging, re, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

log = logging.getLogger("bioscraper.oa_v3")


def _build_session(user_agent="NCBI-BioScraper-Pro/3.5"):
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


def _build_session_no_429_retry(user_agent="NCBI-BioScraper-Pro/3.5"):
    """Like _build_session but does NOT auto-retry 429s. Used for providers
    where we want to see the 429 status code directly (e.g. so a self-throttle
    latch can engage instead of letting urllib3 burn 3 retries silently)."""
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=(500, 502, 503, 504),  # 429 deliberately omitted
                  allowed_methods=frozenset(["GET", "HEAD"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    s.mount("https://", adapter); s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    return s


@dataclass
class OAResult:
    pmid: Optional[str] = None
    doi: Optional[str] = None
    pmcid: Optional[str] = None
    oa_source: str = "none"
    oa_pdf_url: Optional[str] = None
    oa_pdf_fallback_url: Optional[str] = None  # secondary URL to try if primary 404s
    oa_xml_url: Optional[str] = None
    oa_html_url: Optional[str] = None
    oa_license: Optional[str] = None
    oa_status: str = "closed"
    oa_full_text: bool = False
    oa_attempts: list = field(default_factory=list)
    oa_resolved_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def as_flat_dict(self):
        d = asdict(self)
        d["oa_attempts"] = ",".join(d["oa_attempts"])
        return d


# ---------- providers --------------------------------------------------------
class PMCResolver:
    IDCONV = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    OA_FCGI = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"
    BIOC = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"

    def __init__(self, session, ncbi_api_key=None, tool="ncbi-bioscraper", email="research@example.com"):
        self.s = session; self.api_key = ncbi_api_key; self.tool = tool; self.email = email

    def _params(self, **extra):
        p = {"tool": self.tool, "email": self.email}
        if self.api_key: p["api_key"] = self.api_key
        p.update(extra); return p

    def pmid_to_pmcid(self, pmid, doi=None):
        for key, value in (("pmid", pmid), ("doi", doi)):
            if not value: continue
            try:
                r = self.s.get(self.IDCONV,
                               params=self._params(ids=value, idtype=key, format="json"),
                               timeout=6)
                if r.status_code != 200: continue
                records = r.json().get("records", [])
                if records and records[0].get("pmcid"):
                    return records[0]["pmcid"]
            except Exception as e:
                log.debug("idconv (%s=%s) failed: %s", key, value, e)
        return None

    def fetch_bioc_text(self, pmcid):
        """Fetch already-parsed plain text via NCBI's BioC API.

        BioC is one of NCBI's officially endorsed automated retrieval services
        per https://pmc.ncbi.nlm.nih.gov/tools/openftlist/ — preferred over
        PDF scraping for both reliability and TOS compliance.

        Returns concatenated text or None.
        """
        try:
            url = self.BIOC.format(pmcid=pmcid)
            r = self.s.get(url, timeout=15,
                           headers={"Accept": "application/json"})
            if r.status_code != 200: return None
            try:
                data = r.json()
            except Exception:
                return None
            if not isinstance(data, list) or not data:
                return None
            # Walk: collection → documents → passages → text
            chunks = []
            collection = data[0]
            for doc in collection.get("documents", []) or []:
                for pas in doc.get("passages", []) or []:
                    t = (pas.get("text") or "").strip()
                    if t: chunks.append(t)
            text = "\n\n".join(chunks).strip()
            return text or None
        except Exception as e:
            log.debug("bioc fetch failed for %s: %s", pmcid, e)
            return None

    def pmcid_to_oa(self, pmcid):
        """Build candidate URLs for a PMC ID.

        NCBI's FTP-based oa_pdf service is being deprecated (scheduled August
        2026), and PDFs are intermittently 404ing on it already. So we now
        prefer the modern web URL pattern as primary, and only fall back to
        the FTP-derived URL as a last resort.

        URL strategy (in priority order):
          1. https://pmc.ncbi.nlm.nih.gov/articles/PMCxxxxxxx/pdf/   (modern)
          2. Europe PMC fulltextXML URL                              (mirror)
          3. The legacy ftp.ncbi.nlm.nih.gov/pub/pmc/oa_pdf/... URL  (legacy)

        We don't HEAD-check at this stage — that would double the request
        volume. Instead we hand the candidate list back, and the extractor
        tries them in order.
        """
        # Always have the modern reader URL — works even when there's no
        # downloadable PDF (e.g. PMC-hosted but not in OA subset)
        modern_html = "https://pmc.ncbi.nlm.nih.gov/articles/%s/" % pmcid
        modern_pdf  = "https://pmc.ncbi.nlm.nih.gov/articles/%s/pdf/" % pmcid

        try:
            r = self.s.get(self.OA_FCGI, params={"id": pmcid}, timeout=6)
            if r.status_code != 200 or "<error" in r.text.lower():
                # Not in OA subset — return modern reader URL only
                return None
            text = r.text
            ftp_pdf_url = ftp_xml_url = license_str = None
            for line in text.split("<link "):
                if 'format="pdf"' in line and 'href="' in line:
                    ftp_pdf_url = line.split('href="')[1].split('"')[0]
                elif 'format="tgz"' in line and 'href="' in line:
                    ftp_xml_url = line.split('href="')[1].split('"')[0]
            if 'license="' in text:
                license_str = text.split('license="')[1].split('"')[0]
            # Convert ftp:// to https:// for the legacy URL
            if ftp_pdf_url and ftp_pdf_url.startswith("ftp://"):
                ftp_pdf_url = "https://" + ftp_pdf_url[len("ftp://"):]
            if ftp_xml_url and ftp_xml_url.startswith("ftp://"):
                ftp_xml_url = "https://" + ftp_xml_url[len("ftp://"):]
            # Confirmed in OA subset → use modern PDF URL as primary,
            # legacy FTP URL as fallback
            return {
                "pdf_url":      modern_pdf,           # primary
                "pdf_fallback": ftp_pdf_url,           # legacy, may 404
                "xml_url":      ftp_xml_url,
                "license":      license_str,
                "html_url":     modern_html,
            }
        except Exception as e:
            log.debug("oa.fcgi failed for %s: %s", pmcid, e)
        return None

    def resolve(self, result):
        result.oa_attempts.append("pmc")
        if not result.pmcid:
            result.pmcid = self.pmid_to_pmcid(result.pmid, result.doi)
        if not result.pmcid:
            return False
        oa = self.pmcid_to_oa(result.pmcid)
        if not oa:
            # PMC-hosted but not in OA subset — still useful as a reader URL
            result.oa_html_url = "https://pmc.ncbi.nlm.nih.gov/articles/%s/" % result.pmcid
            return False
        result.oa_source = "pmc"
        result.oa_pdf_url = oa.get("pdf_url")
        result.oa_pdf_fallback_url = oa.get("pdf_fallback")
        result.oa_xml_url = oa.get("xml_url")
        result.oa_html_url = oa.get("html_url")
        result.oa_license = oa.get("license")
        result.oa_status = "gold"
        result.oa_full_text = bool(result.oa_pdf_url or result.oa_xml_url)
        return result.oa_full_text


class UnpaywallResolver:
    BASE = "https://api.unpaywall.org/v2/"

    def __init__(self, session, email):
        self.s = session; self.email = email
        self._not_in_index = 0
        self._auth_failed = False  # latched: stop calling once we see a 422

    def resolve(self, result):
        result.oa_attempts.append("unpaywall")
        if self._auth_failed or not result.doi:
            return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"email": self.email}, timeout=6)
            if r.status_code == 404:
                self._not_in_index += 1
                return False
            if r.status_code == 422:
                # Auth/email problem — latch and stop hammering the API
                self._auth_failed = True
                log.warning("Unpaywall 422 — disabling Unpaywall for the rest of this run "
                            "(check USER_EMAIL is a real address).")
                return False
            if r.status_code != 200:
                return False
            d = r.json()
            if not d.get("is_oa"):
                return False
            best = d.get("best_oa_location") or {}
            if not best:
                locs = d.get("oa_locations") or []
                if locs: best = locs[0]
            if not best: return False
            pdf  = best.get("url_for_pdf")
            html = best.get("url_for_landing_page") or best.get("url")
            if not (pdf or html): return False
            result.oa_source = "unpaywall"
            result.oa_pdf_url = pdf
            result.oa_html_url = html
            result.oa_license = best.get("license")
            result.oa_status = d.get("oa_status") or "bronze"
            result.oa_full_text = bool(pdf)
            return bool(pdf or html)
        except Exception as e:
            log.debug("unpaywall failed for %s: %s", result.doi, e)
            return False


class EuropePMCResolver:
    SEARCH = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def __init__(self, session): self.s = session

    def resolve(self, result):
        result.oa_attempts.append("europepmc")
        if result.pmid:
            query = "EXT_ID:%s AND SRC:MED" % result.pmid
        elif result.doi:
            query = 'DOI:"%s"' % result.doi
        else:
            return False
        try:
            r = self.s.get(self.SEARCH, params={
                "query": query, "format": "json",
                "resultType": "core", "pageSize": 1,
            }, timeout=6)
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            if not hits: return False
            h = hits[0]
            if h.get("isOpenAccess") != "Y": return False
            full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
            pdf = next((u["url"] for u in full_text_urls
                        if u.get("documentStyle", "").lower() == "pdf"
                        and u.get("availability", "").lower() == "open access"), None)
            html = next((u["url"] for u in full_text_urls
                         if u.get("documentStyle", "").lower() == "html"), None)
            if not (pdf or html): return False
            result.oa_source = "europepmc"
            result.oa_pdf_url = pdf or result.oa_pdf_url
            result.oa_html_url = html or result.oa_html_url
            result.oa_license = h.get("license") or result.oa_license
            result.oa_status = "gold" if h.get("inEPMC") == "Y" else "green"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("europepmc failed: %s", e)
            return False


class CrossRefOAResolver:
    """Catches recent papers (<30 days) that Unpaywall hasn't ingested yet,
    and any paper with a CC license declared in CrossRef metadata."""
    BASE = "https://api.crossref.org/works/"

    def __init__(self, session, email):
        self.s = session; self.email = email

    def resolve(self, result):
        result.oa_attempts.append("crossref")
        if not result.doi: return False
        try:
            r = self.s.get(self.BASE + quote(result.doi),
                           params={"mailto": self.email}, timeout=6)
            if r.status_code != 200: return False
            msg = r.json().get("message") or {}
            licenses = msg.get("license") or []
            cc = next((L for L in licenses
                       if "creativecommons.org" in (L.get("URL") or "").lower()), None)
            if not cc: return False
            license_url = cc.get("URL", "")
            license_id = None
            if   "/by-nc-nd" in license_url: license_id = "cc-by-nc-nd"
            elif "/by-nc-sa" in license_url: license_id = "cc-by-nc-sa"
            elif "/by-nc"    in license_url: license_id = "cc-by-nc"
            elif "/by-sa"    in license_url: license_id = "cc-by-sa"
            elif "/by"       in license_url: license_id = "cc-by"
            elif "/zero"     in license_url: license_id = "cc0"
            pdf = None
            for link in (msg.get("link") or []):
                if (link.get("content-type") == "application/pdf"
                        and "vor" in (link.get("intended-application", "") or "").lower()):
                    pdf = link.get("URL"); break
            result.oa_source = "crossref"
            result.oa_pdf_url = pdf
            result.oa_html_url = "https://doi.org/" + result.doi
            result.oa_license = license_id or "cc-detected"
            result.oa_status = "gold"
            result.oa_full_text = bool(pdf)
            return True
        except Exception as e:
            log.debug("crossref failed for %s: %s", result.doi, e)
            return False


class SemanticScholarResolver:
    """Semantic Scholar's openAccessPdf field. Free, no key needed at low
    volume (~1 req/sec unauthenticated). Catches a different slice than
    Unpaywall/CrossRef — especially newer papers and CS/biology preprints
    where their crawler picks up author-hosted PDFs.

    Self-rate-limits to 1 req/sec to avoid 429 cascades.
    Skips itself entirely if `result.oa_full_text` is already True from an
    earlier provider — saves quota for records that actually need it.
    Uses its own session that does NOT retry 429s, so the latch can engage
    on the first 429 instead of letting urllib3 burn 3 silent retries.
    """
    BASE = "https://api.semanticscholar.org/graph/v1/paper/"
    MIN_INTERVAL = 1.05  # seconds between calls

    def __init__(self, session=None, email=None):
        # Ignore the shared session — S2 needs the no-retry-on-429 behaviour
        self.s = _build_session_no_429_retry()
        self.email = email
        self._last_call = 0.0
        self._rate_limited = False  # latch after a 429 to stop hammering

    def _lookup(self, identifier):
        if self._rate_limited:
            return None
        # Self-throttle to 1 req/sec
        elapsed = time.time() - self._last_call
        if elapsed < self.MIN_INTERVAL:
            time.sleep(self.MIN_INTERVAL - elapsed)
        self._last_call = time.time()
        try:
            r = self.s.get(self.BASE + identifier,
                           params={"fields": "openAccessPdf,externalIds,isOpenAccess,title"},
                           timeout=6)
            if r.status_code == 429:
                # Latch off — S2 has decided we've had enough. Don't keep retrying.
                self._rate_limited = True
                log.warning("Semantic Scholar 429 — disabling S2 for the rest of this run.")
                return None
            if r.status_code != 200:
                return None
            return r.json()
        except Exception as e:
            # If urllib3 still managed to hit retry exhaustion on 429s
            # (shouldn't with our no-retry session, but belt-and-braces),
            # treat that as a hard latch too.
            err_str = str(e).lower()
            if "429" in err_str or "too many" in err_str:
                self._rate_limited = True
                log.warning("Semantic Scholar repeated 429 — disabling S2 for the rest of this run.")
            else:
                log.debug("semanticscholar lookup %s failed: %s", identifier, e)
            return None

    def resolve(self, result):
        # Skip entirely if an earlier provider already gave us a PDF
        if result.oa_full_text:
            return False
        result.oa_attempts.append("semanticscholar")
        d = None
        if result.doi:
            d = self._lookup("DOI:" + result.doi)
        if not d and result.pmid:
            d = self._lookup("PMID:" + result.pmid)
        if not d:
            return False
        oa_pdf = d.get("openAccessPdf") or {}
        url = oa_pdf.get("url")
        if not url:
            return False
        result.oa_source = "semanticscholar"
        result.oa_pdf_url = url
        result.oa_html_url = result.oa_html_url or url
        result.oa_license = oa_pdf.get("license") or result.oa_license or "s2-detected"
        if result.oa_status == "closed":
            result.oa_status = "green"
        result.oa_full_text = True
        return True


# OAButtonResolver removed in v3.4: OA.Works retired the openaccessbutton.org
# API on 2025-11-18. Calls were 100% failing with DNS errors. If you want a
# similar drop-in replacement, CORE (api.core.ac.uk, free key) or Semantic
# Scholar's openAccessPdf field both work and follow the same `resolve(result)`
# contract.


class PreprintResolver:
    """Direct lookup for 10.1101/* DOIs (bioRxiv/medRxiv)."""
    def __init__(self, session): self.s = session


    def resolve(self, result):
        result.oa_attempts.append("preprint")
        if not result.doi or not result.doi.startswith("10.1101/"):
            return False
        for host in ("www.biorxiv.org", "www.medrxiv.org"):
            url = "https://%s/content/%s.full.pdf" % (host, result.doi)
            try:
                r = self.s.head(url, timeout=8, allow_redirects=True)
                if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
                    result.oa_source = "preprint"
                    result.oa_pdf_url = url
                    result.oa_html_url = url.rsplit(".full.pdf", 1)[0] + ".full"
                    result.oa_status = "green"
                    result.oa_full_text = True
                    return True
            except Exception:
                continue
        return False


class PreprintTitleSearchResolver:
    """Last-resort: search bioRxiv/medRxiv by title and accept matches with
    high token-overlap similarity. Useful when a peer-reviewed paper has
    a preprint twin under a different DOI."""
    BIORXIV_API = "https://api.biorxiv.org/details/{server}/{date_or_doi}/0/json"
    SEARCH = "https://www.biorxiv.org/search/{query}"  # HTML, not used directly

    def __init__(self, session, min_similarity=0.75):
        self.s = session
        self.min_similarity = min_similarity

    @staticmethod
    def _norm_tokens(s):
        s = (s or "").lower()
        s = re.sub(r"[^a-z0-9 ]+", " ", s)
        return set(t for t in s.split() if len(t) > 2)

    def _similarity(self, a, b):
        ta, tb = self._norm_tokens(a), self._norm_tokens(b)
        if not ta or not tb: return 0.0
        return len(ta & tb) / len(ta | tb)  # Jaccard

    def resolve(self, result, candidate_title=None):
        result.oa_attempts.append("preprint_search")
        if not candidate_title: return False
        # bioRxiv has no native title-search API; we use Europe PMC restricted
        # to preprint sources, which DOES index bioRxiv/medRxiv records.
        try:
            r = self.s.get(
                "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                params={"query": 'TITLE:"%s" AND (SRC:PPR)' % candidate_title[:200],
                        "format": "json", "resultType": "core", "pageSize": 5},
                timeout=6,
            )
            if r.status_code != 200: return False
            hits = (r.json().get("resultList") or {}).get("result") or []
            for h in hits:
                sim = self._similarity(candidate_title, h.get("title"))
                if sim < self.min_similarity: continue
                doi = h.get("doi")
                full_text_urls = (h.get("fullTextUrlList") or {}).get("fullTextUrl") or []
                pdf = next((u["url"] for u in full_text_urls
                            if u.get("documentStyle", "").lower() == "pdf"), None)
                html = next((u["url"] for u in full_text_urls
                             if u.get("documentStyle", "").lower() == "html"), None)
                if not (pdf or html): continue
                result.oa_source = "preprint_search"
                result.oa_pdf_url = pdf or result.oa_pdf_url
                result.oa_html_url = html or result.oa_html_url
                result.oa_status = "green"
                result.oa_full_text = bool(pdf)
                return True
        except Exception as e:
            log.debug("preprint_search failed: %s", e)
        return False


# ---------- orchestrator -----------------------------------------------------
class OAFullTextResolver:
    def __init__(self, email, ncbi_api_key=None,
                 enable_pmc=True, enable_unpaywall=True,
                 enable_europepmc=True, enable_crossref=True,
                 enable_semanticscholar=True,
                 enable_oabutton=False,  # kept for back-compat; ignored (API retired)
                 enable_preprint=True,
                 enable_preprint_search=False,
                 polite_delay_s=0.0):
        self.session = _build_session()
        self.polite_delay = polite_delay_s
        self.providers = []
        if enable_pmc:
            self.providers.append(PMCResolver(self.session, ncbi_api_key, email=email))
        if enable_unpaywall:
            self.providers.append(UnpaywallResolver(self.session, email=email))
        if enable_europepmc:
            self.providers.append(EuropePMCResolver(self.session))
        if enable_crossref:
            self.providers.append(CrossRefOAResolver(self.session, email=email))
        if enable_semanticscholar:
            self.providers.append(SemanticScholarResolver(self.session, email=email))
        # OA Button removed in v3.4 (api.openaccessbutton.org retired 2025-11-18)
        if enable_preprint:
            self.providers.append(PreprintResolver(self.session))
        # Title-search resolver is held separately because it needs a title arg
        self.preprint_search = (PreprintTitleSearchResolver(self.session)
                                if enable_preprint_search else None)

    def resolve_one(self, pmid=None, doi=None, pmcid=None, title=None):
        result = OAResult(pmid=pmid, doi=doi, pmcid=pmcid)
        for provider in self.providers:
            try:
                if provider.resolve(result) and result.oa_full_text:
                    break
            except Exception as e:
                log.warning("%s raised: %s", type(provider).__name__, e)
            if self.polite_delay:
                time.sleep(self.polite_delay)
        # Title-similarity fallback only if we still don't have full text
        if (self.preprint_search is not None
                and not result.oa_full_text
                and title):
            try:
                self.preprint_search.resolve(result, candidate_title=title)
            except Exception as e:
                log.warning("PreprintTitleSearchResolver raised: %s", e)
        return result

    def enrich_dataframe(self, df, pmid_col="pmid", doi_col="doi",
                         pmcid_col=None, title_col=None, show_progress=True):
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(df.iterrows(), total=len(df),
                            desc="OA resolve", disable=not show_progress)
        except ImportError:
            iterator = df.iterrows()

        rows = []
        import pandas as pd
        for _, row in iterator:
            def _get(col):
                if col and col in df.columns:
                    v = row[col]
                    if v is None: return None
                    try:
                        if pd.isna(v): return None
                    except Exception: pass
                    s = str(v).strip()
                    return s if s and s.lower() != "nan" else None
                return None
            rows.append(self.resolve_one(
                pmid=_get(pmid_col),
                doi=_get(doi_col),
                pmcid=_get(pmcid_col),
                title=_get(title_col),
            ).as_flat_dict())

        oa_df = pd.DataFrame(rows)
        # Don't re-emit identifier cols already present on the input df, otherwise
        # a second pass through enrich_dataframe will collide on pmid/doi/pmcid.
        for c in ("pmid", "doi", "pmcid"):
            if c in oa_df.columns and c in df.columns:
                oa_df = oa_df.drop(columns=[c])
        for c in list(df.columns):
            if c.startswith("oa_") or c.startswith("up_"):
                df = df.drop(columns=[c])
        return df.reset_index(drop=True).join(oa_df.reset_index(drop=True))

    def coverage_report(self, df):
        total = len(df)
        if total == 0: return {"total": 0}
        return {
            "total": total,
            "full_text_pct": round(100 * df["oa_full_text"].sum() / total, 1),
            "any_oa_link_pct": round(100 * (df["oa_source"] != "none").sum() / total, 1),
            "by_source": df["oa_source"].value_counts().to_dict(),
            "by_status": df["oa_status"].value_counts().to_dict(),
        }
'''

# Write it ONCE, force a clean import
with open(_MODULE_PATH, "w", encoding="utf-8") as f:
    f.write(_MODULE_SRC)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
for _mod in [m for m in sys.modules if m.startswith("oa_fulltext_v3")]:
    del sys.modules[_mod]

from oa_fulltext_v3 import OAFullTextResolver, UnpaywallResolver  # noqa: E402

# ---- 2. Build resolver with sane provider gating ---------------------------
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("bioscraper.oa_v3").setLevel(logging.WARNING)  # quiet the per-record DEBUG noise

oa = OAFullTextResolver(
    email=USER_EMAIL or "research@example.com",
    ncbi_api_key=NCBI_API_KEY,
    enable_pmc=True,
    enable_unpaywall=ENRICH_UNPAYWALL and _email_is_real,   # << the fix
    enable_europepmc=True,
    enable_crossref=_email_is_real,                          # CrossRef wants mailto too
    # OA Button retired 2025-11-18, kwarg dropped
    enable_preprint=True,
    enable_preprint_search=PREPRINT_TITLE_SEARCH,
)

# ---- 3. Enrich -------------------------------------------------------------
# Self-heal: if `df` was knocked out of the namespace by a prior error or
# kernel reset, rebuild it from the SQLite cache from earlier scrapes.
def _rebuild_df_from_cache():
    """Rebuild a minimal df (pmid/doi/title) from the pubmed_cache SQLite table.
    Only enough columns for Cell 7 to operate. Does not re-fetch from PubMed."""
    import sqlite3, json as _json
    import pandas as _pd

    candidate_paths = []
    if "CACHE_DB" in globals():
        candidate_paths.append(globals()["CACHE_DB"])
    if "OUTPUT_DIR" in globals():
        candidate_paths.append(os.path.join(globals()["OUTPUT_DIR"], "_cache.sqlite"))
    # Common defaults if config globals also got wiped
    candidate_paths += [
        "/content/output/_cache.sqlite",
        "/content/drive/MyDrive/NCBI_Research/crispr_cancer_v2/_cache.sqlite",
    ]

    db_path = next((p for p in candidate_paths if p and os.path.exists(p)), None)
    if not db_path:
        return None, "no cache file found at: " + ", ".join(str(p) for p in candidate_paths)

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.execute("SELECT pmid, record_json FROM pubmed_cache")
        rows = []
        for pmid, blob in cursor.fetchall():
            try:
                rec = _json.loads(blob) if blob else {}
            except Exception:
                rec = {}
            doi = (rec.get("LID") or rec.get("AID") or "")
            if isinstance(doi, list): doi = doi[0] if doi else ""
            doi = str(doi).strip()
            # Strip trailing "[doi]" tag PubMed sometimes attaches
            doi = re.sub(r"\s*\[doi\]\s*$", "", doi, flags=re.IGNORECASE).strip()
            # Keep only DOIs that look like DOIs (start with 10.)
            if doi and not doi.startswith("10."):
                m = re.search(r"\b(10\.\d{4,9}/[^\s]+)", doi)
                doi = m.group(1) if m else ""
            rows.append({
                "pmid":  str(pmid),
                "doi":   doi,
                "title": str(rec.get("TI", "")),
                "abstract": str(rec.get("AB", "")),
                "pmcid": str(rec.get("PMC", "") or ""),
            })
        conn.close()
        if not rows:
            return None, f"cache at {db_path} has 0 records"
        return _pd.DataFrame(rows), f"loaded {len(rows)} records from {db_path}"
    except Exception as e:
        return None, f"cache read failed at {db_path}: {e}"


def _fresh_pubmed_scrape(query, max_results=50, start_year=None, end_year=None,
                         email=None, api_key=None):
    """Minimal self-contained PubMed scrape using NCBI E-utilities REST API.
    Only pulls the fields needed for OA resolution: pmid, doi, title, abstract,
    pmcid. No Biopython dependency. Returns a DataFrame or None on failure.
    """
    import urllib.parse, requests as _rq
    import pandas as _pd
    from xml.etree import ElementTree as ET

    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    session = _rq.Session()
    session.headers.update({"User-Agent": "NCBI-BioScraper-Pro/3.5.5"})

    # Build query with date filter if given
    full_query = query
    if start_year and end_year:
        full_query += f" AND ({start_year}[PDAT]:{end_year}[PDAT])"

    # 1. esearch — get PMIDs
    esearch_params = {"db": "pubmed", "term": full_query, "retmax": max_results,
                      "retmode": "json"}
    if email:   esearch_params["email"] = email
    if api_key: esearch_params["api_key"] = api_key

    try:
        r = session.get(base + "esearch.fcgi", params=esearch_params, timeout=15)
        r.raise_for_status()
        ids = r.json().get("esearchresult", {}).get("idlist", []) or []
    except _rq.exceptions.HTTPError as e:
        status = e.response.status_code if e.response is not None else "?"
        print(f"   ❌ esearch failed (HTTP {status}): {e}")
        if status == 429:
            print("       NCBI is rate-limiting you. Wait 60s and retry.")
        elif status == 403:
            print("       NCBI is blocking the source IP. Set NCBI_API_KEY in Cell 2")
            print("       to authenticate, or wait if you've been hitting rate limits.")
        return None
    except Exception as e:
        print(f"   ❌ esearch failed: {type(e).__name__}: {e}")
        return None
    if not ids:
        print(f"   ❌ esearch returned 0 PMIDs for query: {full_query!r}")
        return None
    print(f"   esearch: {len(ids)} PMIDs")

    # 2. efetch — get full records (XML for structured DOI/PMC extraction)
    efetch_params = {"db": "pubmed", "id": ",".join(ids),
                     "rettype": "xml", "retmode": "xml"}
    if email:   efetch_params["email"] = email
    if api_key: efetch_params["api_key"] = api_key

    try:
        r = session.get(base + "efetch.fcgi", params=efetch_params, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
    except Exception as e:
        print(f"   ❌ efetch failed: {e}")
        return None

    rows = []
    for article in root.iter("PubmedArticle"):
        pmid_elem = article.find(".//PMID")
        pmid = pmid_elem.text if pmid_elem is not None else ""

        title_elem = article.find(".//ArticleTitle")
        title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

        # Abstract may be split across multiple AbstractText elements
        abstract_parts = []
        for at in article.iter("AbstractText"):
            t = "".join(at.itertext()).strip()
            if t: abstract_parts.append(t)
        abstract = " ".join(abstract_parts)

        # DOI lives in ArticleIdList as <ArticleId IdType="doi">
        doi = ""; pmcid = ""
        for aid in article.iter("ArticleId"):
            id_type = (aid.get("IdType") or "").lower()
            val = (aid.text or "").strip()
            if id_type == "doi" and not doi: doi = val
            elif id_type == "pmc" and not pmcid: pmcid = val

        rows.append({
            "pmid":     str(pmid),
            "doi":      doi,
            "title":    title,
            "abstract": abstract,
            "pmcid":    pmcid,
        })

    if not rows:
        print(f"   ❌ efetch returned 0 parsable articles")
        return None
    return _pd.DataFrame(rows)


if "df" not in globals() or globals().get("df") is None:
    print("⚠️  `df` not in scope — attempting recovery…\n")
    import pandas as pd

    # Attempt 1: rebuild from SQLite cache (free, instant)
    print("   [1/2] Trying SQLite cache…")
    rebuilt, msg = _rebuild_df_from_cache()
    print(f"         {msg}")

    if rebuilt is not None and len(rebuilt) > 0:
        df = rebuilt
        print(f"   ✅ Rebuilt from cache: {len(df)} records")
    else:
        # Attempt 2: fresh PubMed scrape using config from Cell 2
        query        = globals().get("SEARCH_QUERY",  "CRISPR cancer therapy")
        max_results  = globals().get("MAX_RESULTS",   50)
        start_year   = globals().get("START_YEAR",    None)
        end_year     = globals().get("END_YEAR",      None)
        print(f"\n   [2/2] Cache empty — running fresh PubMed scrape:")
        print(f"         query='{query}' | max={max_results} | years={start_year}-{end_year}")
        rebuilt = _fresh_pubmed_scrape(query=query, max_results=max_results,
                                       start_year=start_year, end_year=end_year,
                                       email=USER_EMAIL if _email_is_real else None,
                                       api_key=NCBI_API_KEY)
        if rebuilt is None or len(rebuilt) == 0:
            print("\n❌ Could not recover `df` from either cache or fresh scrape.")
            print("   Check your network and SEARCH_QUERY config in Cell 2.")
            raise RuntimeError("df missing and recovery failed")
        df = rebuilt
        print(f"   ✅ Fresh scrape complete: {len(df)} records")

    print(f"   Columns: {list(df.columns)}")
    print(f"   With DOI:  {(df['doi'] != '').sum()}/{len(df)}")
    print(f"   With PMCID:{(df['pmcid'] != '').sum()}/{len(df)}")
    print()

# Use 'title' column if it exists, for the preprint title-similarity fallback
_title_col = "title" if "title" in df.columns else None
df = df.drop(columns=[c for c in df.columns if c.startswith("oa_") or c.startswith("up_")],
             errors="ignore")
df = oa.enrich_dataframe(df, pmid_col="pmid", doi_col="doi", title_col=_title_col)

# ---- 4. Coverage report ----------------------------------------------------
report = oa.coverage_report(df)
print("\n📊 OA full-text resolution v3.5.7 MAX (%d records)" % report["total"])
print("   Full-text yield: %s%%" % report["full_text_pct"])
print("   Any OA link:     %s%%" % report["any_oa_link_pct"])
print("   By source:       %s" % report["by_source"])
print("   By status:       %s" % report["by_status"])

unpaywall_obj = next((p for p in oa.providers if isinstance(p, UnpaywallResolver)), None)
if unpaywall_obj is not None:
    n_miss = unpaywall_obj._not_in_index
    print(f"\n   Unpaywall index misses (404s): {n_miss}/{len(df)}")
    print( "   -> these are typically <30-day-old DOIs not yet ingested by Unpaywall")
    if unpaywall_obj._auth_failed:
        print("   ⚠️  Unpaywall auth failed mid-run — fix USER_EMAIL and re-run this cell.")
elif ENRICH_UNPAYWALL and not _email_is_real:
    print("\n   Unpaywall: SKIPPED (placeholder email).")


# ============================================================================
# CELL 7b — Download PDFs and extract plain text
# Controlled by EXTRACT_PDF_TEXT flag. ON by default in v3.5 MAX.
# Uses pypdf only (no GPU, no OCR). Caps at PDF_TEXT_MAX_RECORDS for safety.
# ============================================================================
print()
print("─" * 72)
print(f"  PDF text extraction: {'ON' if EXTRACT_PDF_TEXT else 'OFF'}  "
      f"(EXTRACT_PDF_TEXT = {EXTRACT_PDF_TEXT})")
if EXTRACT_PDF_TEXT:
    print(f"  Cap: {PDF_TEXT_MAX_RECORDS} PDFs, "
          f"{PDF_TEXT_MAX_BYTES // 1_000_000} MB each")
print("─" * 72)

if EXTRACT_PDF_TEXT:
    try:
        import pypdf  # noqa: F401
    except ImportError:
        print("Installing pypdf for text extraction...")
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
        import pypdf

    import io, requests as _rq
    from collections import Counter as _Counter
    from pypdf import PdfReader

    # Browser-style User-Agent — many publishers block obvious scrapers but
    # serve OA content fine to a normal-looking browser fetch. This is OA
    # content the publisher has affirmatively licensed for free reading.
    _PDF_UA = ("Mozilla/5.0 (X11; Linux x86_64) "
               "AppleWebKit/537.36 (KHTML, like Gecko) "
               "Chrome/124.0.0.0 Safari/537.36")
    _PDF_HEADERS = {
        "User-Agent": _PDF_UA,
        "Accept": "application/pdf,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    # Track precise failure reasons so we can SEE why extraction is failing
    # rather than guessing. This is the key change in v3.5.3.
    _failure_reasons = []   # list of (idx, source, url, reason, detail)

    def _extract_pdf_text(url, max_bytes=PDF_TEXT_MAX_BYTES, idx=None, source=None):
        if not url:
            _failure_reasons.append((idx, source, url, "no_url", ""))
            return None
        try:
            with _rq.get(url, stream=True, timeout=20,
                         headers=_PDF_HEADERS, allow_redirects=True) as r:
                if r.status_code != 200:
                    _failure_reasons.append(
                        (idx, source, url, f"http_{r.status_code}",
                         f"final_url={r.url}"))
                    return None
                ctype = (r.headers.get("content-type") or "").lower()
                final_url = r.url
                if "pdf" not in ctype and not final_url.lower().endswith(".pdf"):
                    # Likely redirected to a login wall or HTML reader page
                    _failure_reasons.append(
                        (idx, source, url, "not_pdf", f"ctype={ctype} final={final_url}"))
                    return None
                buf = io.BytesIO()
                total = 0
                for chunk in r.iter_content(chunk_size=65536):
                    if not chunk: continue
                    buf.write(chunk); total += len(chunk)
                    if total > max_bytes:
                        _failure_reasons.append(
                            (idx, source, url, "too_big", f"{total:,} bytes"))
                        return None
                buf.seek(0)
                try:
                    reader = PdfReader(buf)
                except Exception as e:
                    _failure_reasons.append(
                        (idx, source, url, "pdf_parse_error", str(e)[:120]))
                    return None
                pages = []
                for page in reader.pages:
                    try: pages.append(page.extract_text() or "")
                    except Exception: pass
                text = "\n\n".join(pages).strip()
                text = re.sub(r"[ \t]+", " ", text)
                text = re.sub(r"\n{3,}", "\n\n", text)
                if not text:
                    _failure_reasons.append(
                        (idx, source, url, "empty_text", f"{total:,} bytes downloaded"))
                    return None
                return text
        except _rq.exceptions.Timeout:
            _failure_reasons.append((idx, source, url, "timeout", ""))
        except _rq.exceptions.SSLError as e:
            _failure_reasons.append((idx, source, url, "ssl_error", str(e)[:120]))
        except Exception as e:
            _failure_reasons.append((idx, source, url, "exception", str(e)[:120]))
        return None

    # Helper: fetch PMC full text via the BioC API (NCBI's officially endorsed
    # automated retrieval service for PMC content). Returns text or None.
    _BIOC_URL = ("https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/"
                 "pmcoa.cgi/BioC_json/{pmcid}/unicode")
    def _fetch_bioc_text(pmcid, idx=None):
        if not pmcid: return None
        url = _BIOC_URL.format(pmcid=str(pmcid))
        try:
            r = _rq.get(url, timeout=15,
                        headers={"User-Agent": _PDF_UA,
                                 "Accept": "application/json"})
            if r.status_code != 200:
                _failure_reasons.append(
                    (idx, "pmc_bioc", url, f"http_{r.status_code}", ""))
                return None
            try: data = r.json()
            except Exception as e:
                _failure_reasons.append(
                    (idx, "pmc_bioc", url, "bioc_parse_error", str(e)[:120]))
                return None
            if not isinstance(data, list) or not data:
                _failure_reasons.append(
                    (idx, "pmc_bioc", url, "bioc_empty", ""))
                return None
            chunks = []
            for doc in (data[0].get("documents") or []):
                for pas in (doc.get("passages") or []):
                    t = (pas.get("text") or "").strip()
                    if t: chunks.append(t)
            text = "\n\n".join(chunks).strip()
            return text or None
        except Exception as e:
            _failure_reasons.append(
                (idx, "pmc_bioc", url, "exception", str(e)[:120]))
            return None

    # Eligibility: any record with oa_full_text=True is in the queue. For PMC
    # records we use BioC API (no PDF needed); for non-PMC we still need a
    # PDF URL.
    is_pmc = df["oa_source"] == "pmc"
    has_pdf_url = df["oa_pdf_url"].notna() & (df["oa_pdf_url"] != "")
    eligible_mask = df["oa_full_text"] & (is_pmc | has_pdf_url)
    eligible = df[eligible_mask].copy()
    priority_order = ["pmc", "preprint", "preprint_search", "europepmc",
                      "unpaywall", "semanticscholar", "crossref"]
    eligible["_prio"] = eligible["oa_source"].map(
        {s: i for i, s in enumerate(priority_order)}).fillna(99)
    eligible = eligible.sort_values("_prio").head(PDF_TEXT_MAX_RECORDS)

    # Diagnostic context
    n_full_text  = int(df["oa_full_text"].sum())
    n_with_pdf   = int(has_pdf_url.sum())
    n_pmc        = int(is_pmc.sum())

    if len(eligible) == 0:
        print("\n  ⚠️  No records eligible for extraction.")
        df["oa_full_text_extracted"] = None
        df["oa_full_text_chars"] = 0
    else:
        print(f"\n📄 Extraction queue ({len(eligible)} of {n_full_text} full-text records):")
        print(f"   PMC records (use BioC API): {n_pmc}")
        print(f"   Records with a PDF URL:     {n_with_pdf}")
        print(f"   By source in queue: {dict(eligible['oa_source'].value_counts())}")

        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(eligible.iterrows(), total=len(eligible), desc="Text extract")
        except ImportError:
            it = eligible.iterrows()

        texts = {}
        n_via_bioc = 0
        n_via_pdf = 0
        n_fallback_used = 0
        for idx, row in it:
            t = None
            # Strategy 1: PMC source → BioC API (preferred — official, fast, reliable)
            if row["oa_source"] == "pmc" and row.get("pmcid"):
                t = _fetch_bioc_text(row["pmcid"], idx=idx)
                if t: n_via_bioc += 1
            # Strategy 2: PDF download (for non-PMC OR PMC with no BioC coverage)
            if not t and row.get("oa_pdf_url"):
                t = _extract_pdf_text(row["oa_pdf_url"], idx=idx,
                                       source=row["oa_source"])
                if t: n_via_pdf += 1
            # Strategy 3: legacy FTP fallback URL
            if not t:
                fb = row.get("oa_pdf_fallback_url")
                if fb and isinstance(fb, str) and fb.strip():
                    t = _extract_pdf_text(fb, idx=idx,
                                          source=row["oa_source"] + "_fallback")
                    if t:
                        n_fallback_used += 1
            if t: texts[idx] = t

        df["oa_full_text_extracted"] = df.index.map(texts).where(df.index.isin(texts), None)
        df["oa_full_text_chars"]     = df["oa_full_text_extracted"].fillna("").str.len()
        n_extracted = int(df["oa_full_text_chars"].gt(0).sum())
        median_chars = int(df.loc[df["oa_full_text_chars"] > 0, "oa_full_text_chars"].median() or 0)
        print(f"\n   Extracted: {n_extracted}/{len(eligible)} records")
        print(f"     via BioC API (PMC):  {n_via_bioc}")
        print(f"     via PDF download:    {n_via_pdf}")
        if n_fallback_used:
            print(f"     via fallback URL:    {n_fallback_used}")
        if n_extracted:
            print(f"   Median length: {median_chars:,} chars (~{median_chars//5_000} pages of plain text)")
            print(f"   Total content captured: {int(df['oa_full_text_chars'].sum()):,} chars")

        # Per-source success rate — tells us which providers gave usable PDFs
        succ_by_src = df.loc[df["oa_full_text_chars"] > 0, "oa_source"].value_counts().to_dict()
        attempted_by_src = eligible["oa_source"].value_counts().to_dict()
        print(f"\n   Per-source extraction success:")
        for src in attempted_by_src:
            ok = succ_by_src.get(src, 0); tot = attempted_by_src[src]
            print(f"     {src:18s} {ok}/{tot}")

        # Failure-reason breakdown — the diagnostic that was missing
        if _failure_reasons:
            reason_counts = _Counter(fr[3] for fr in _failure_reasons)
            print(f"\n   Failure reasons ({len(_failure_reasons)} total):")
            for reason, count in reason_counts.most_common():
                print(f"     {reason:18s} {count}")
            # Show first few examples per reason category
            print(f"\n   Failure examples (up to 5):")
            for fr in _failure_reasons[:5]:
                idx_, src_, url_, reason_, detail_ = fr
                short_url = url_[:60] + "…" if url_ and len(url_) > 60 else (url_ or "")
                print(f"     [{src_}] {reason_}: {short_url}")
                if detail_: print(f"       └─ {detail_[:140]}")

        # Stash the failure log on df for post-hoc inspection
        df.attrs["pdf_extract_failures"] = _failure_reasons

        print(f"\n   Records with ABSTRACT only:        {len(df) - n_extracted}")
        print(f"   Records with FULL TEXT extracted:  {n_extracted}")
        print(f"   -> use df['oa_full_text_extracted'] for downstream NLP / LLM input")
        print(f"   -> use df.attrs['pdf_extract_failures'] to inspect failed downloads")


# ============================================================================
# CELL 7c — Optional: Groq LLM summaries + key findings
# Off by default. Requires:  pip install groq   AND   os.environ["GROQ_API_KEY"]
# Operates on EXTRACTED full text if available, else on the abstract.
# ============================================================================
if GROQ_SUMMARIES:
    api_key = os.environ.get("GROQ_API_KEY", "").strip()
    if not api_key:
        print("\n⚠️  GROQ_SUMMARIES=True but GROQ_API_KEY not set in environment. Skipping.")
    else:
        try:
            from groq import Groq  # noqa: F401
        except ImportError:
            print("Installing groq client…")
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
        from groq import Groq

        client = Groq(api_key=api_key)

        SYSTEM_PROMPT = (
            "You are summarising a biomedical paper for a domain expert in "
            "molecular biology, genomics, and computational biology. "
            "Return STRICT JSON with exactly these keys: "
            '{"tldr": str (<=40 words), '
            '"key_findings": list[str] (3-5 bullets, each <=25 words), '
            '"methods": str (<=30 words), '
            '"caveats": str (<=25 words, or empty string if none)}. '
            "Stay concrete. Cite figures or numbers when they appear in the source. "
            "Do not invent details. Output ONLY the JSON object."
        )

        def _summarise_one(text, model=GROQ_MODEL, max_chars=24000):
            text = (text or "").strip()
            if not text: return None
            payload = text[:max_chars]
            try:
                resp = client.chat.completions.create(
                    model=model,
                    temperature=0.2,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": payload},
                    ],
                )
                return resp.choices[0].message.content
            except Exception as e:
                logging.getLogger("bioscraper.oa_v3").warning("Groq summarise failed: %s", e)
                return None

        # Choose source text: extracted full text > abstract
        has_fulltext = ("oa_full_text_extracted" in df.columns)
        candidates = df.copy()
        if has_fulltext:
            candidates["_src"] = candidates["oa_full_text_extracted"].fillna(
                candidates.get("abstract", ""))
        else:
            candidates["_src"] = candidates.get("abstract", "")
        candidates = candidates[candidates["_src"].fillna("").str.len() > 200]
        candidates = candidates.head(GROQ_MAX_RECORDS)

        print(f"\n🤖 Summarising {len(candidates)} records via Groq ({GROQ_MODEL})…")
        try:
            from tqdm.auto import tqdm as _tqdm
            it = _tqdm(candidates.iterrows(), total=len(candidates), desc="LLM summarise")
        except ImportError:
            it = candidates.iterrows()

        summaries = {}
        for idx, row in it:
            s = _summarise_one(row["_src"])
            if s: summaries[idx] = s

        df["llm_summary_json"] = df.index.map(summaries).where(df.index.isin(summaries), None)
        # Lightly parse for convenience
        import json as _json
        def _safe_json(s):
            try: return _json.loads(s) if s else None
            except Exception: return None
        parsed = df["llm_summary_json"].map(_safe_json)
        df["llm_tldr"]         = parsed.map(lambda d: (d or {}).get("tldr"))
        df["llm_key_findings"] = parsed.map(lambda d: " | ".join((d or {}).get("key_findings") or []))
        df["llm_methods"]      = parsed.map(lambda d: (d or {}).get("methods"))
        df["llm_caveats"]      = parsed.map(lambda d: (d or {}).get("caveats"))

        n_ok = df["llm_tldr"].notna().sum()
        print(f"   Summaries produced: {n_ok}/{len(candidates)}")


# ============================================================================
# Notes that are NOT in the code, on purpose:
#
# 1. Sci-Hub fallback was suggested in the feedback. I deliberately did not
#    add it. Sci-Hub redistributes copyrighted papers without authorisation;
#    bundling it into a notebook you plan to publish would expose you (and
#    anyone forking the repo) to DMCA/takedown risk and would get the repo
#    flagged on GitHub. The legitimate stack above (PMC + Unpaywall +
#    Europe PMC + CrossRef + bioRxiv) is what actually gets you to the
#    60–70% yield range your reviewer mentioned, AS LONG AS USER_EMAIL is
#    set. If you want one more legitimate boost, add CORE (api.core.ac.uk,
#    free key) or Semantic Scholar's openAccessPdf field — both are
#    drop-in providers using the same `resolve(result)` contract.
#
# 2. Streamlit dashboard: that's a separate deliverable, not a Cell 7 fix.
#    The right move is a `streamlit_app.py` in the repo root that loads
#    the parquet/csv this notebook produces. Happy to write it as a
#    follow-up — it shouldn't live inside the scraper notebook.
#
# 3. OA Button: was a useful free aggregator but OA.Works shut it down on
#    2025-11-18. The domain still exists for the website redirect but the
#    API DNS record is gone, hence the NameResolutionErrors. Removed in v3.4.
# ============================================================================